In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import igraph
import leidenalg

In [ ]:

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy
import os
import scipy.io as sio
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl


from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator



In [ ]:
#adata = ad.read_h5ad('/scratch/users/edtng/Hippo_timecourse/Hippo:eTN8_GEX_sub1/kb_all_main_raw/counts_filtered/adata.h5ad')

In [ ]:
#adata

In [ ]:
# 1. Define the base path with a placeholder {} for the sub-folder number
base_path = '/scratch/users/edtng/Hippo_timecourse/Hippo:eTN8_GEX_sub{}/kb_all_main_raw/counts_filtered/adata.h5ad'

# 2. Create an empty dictionary to store the loaded AnnData objects
adatas = {}

# 3. Loop through 1 to 8 to load each file
for i in range(1, 9):
    # Format the path for the current sub-folder
    current_path = base_path.format(i)
    
    # Read the data
    print(f"Loading {current_path}...")
    adata = sc.read_h5ad(current_path)
    
    # Store it in the dictionary with the key "s1", "s2", etc.
    dict_key = f"s{i}"
    adatas[dict_key] = adata

# 4. Concatenate all 8 objects
# Passing the dictionary automatically maps the keys ("s1", "s2") to the batches
adata_combined = ad.concat(
    adatas,
    label="sublibID",  # Creates a new column in .obs to track which sample the cell came from
    index_unique="_",      # Joins the original barcode and the dictionary key with an underscore
    join="outer",           # Ensures no genes are dropped if some matrices have slightly different gene lists
    merge="same",          
    uns_merge="same",
)

print("\nConcatenation complete!")
print("New obs_names look like:", adata_combined.obs_names[:5].tolist())

In [ ]:
adata_combined.shape

In [ ]:
adata_combined

In [ ]:
adata = adata_combined

In [ ]:
adata.var["gene"]

In [ ]:
adata.obs['pct_counts_mt']

In [ ]:
sc.settings.verbosity = 1 # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=300, figsize=(5,4), format='png')


### Reading in data

### Cell QC

In [ ]:
# Filter the data
adata = adata[adata.obs.n_genes_by_counts < 12000,:]
adata = adata[adata.obs.n_genes_by_counts > 1200,:]
adata = adata[adata.obs.total_counts < 50000,:]
adata = adata[adata.obs.total_counts > 1200,:]
adata = adata[adata.obs.pct_counts_mt < 10,:]
adata = adata[adata.obs.biological_sample != 'empty',:]
adata.shape # Checking number of cells remaining

In [ ]:
np.median(adata.obs.total_counts)

In [ ]:
# Scanpy will prepend the string in the save argument with "violin"
# and save it to our figure directory defined in the first step.
sc.pl.violin(adata, ['n_genes_by_counts'], save='_n_genes', jitter=0.4)
sc.pl.violin(adata, ['total_counts'], save='_total_counts', jitter=0.4)
sc.pl.violin(adata, ['pct_counts_mt'], save='_mito_pct', jitter=0.4)

In [ ]:
np.max(adata.obs.pct_counts_mt)

In [ ]:
#sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', save='_gene_vs_transcript_counts')
#print('median transcript count per cell: ' + str(adata.obs['tscp_count'].median(0)))
#print('median gene count per cell: ' + str(adata.obs['gene_count'].median(0)))

In [ ]:
# 1. Filter out the cells where the sample is "empty"
# We use .copy() to ensure the result is an independent AnnData object and not just a view
adata = adata[adata.obs['biological_sample'] != 'empty']

# 2. Remove "empty" from the list of categories in the metadata
# This prevents "empty" from showing up with 0 counts in future plots
adata.obs['biological_sample'] = adata.obs['biological_sample'].cat.remove_unused_categories()

# 3. Verify the result
print(adata.obs['biological_sample'].value_counts())

In [ ]:
import matplotlib.pyplot as plt

# 1. Count the number of cells in each category
cell_counts = adata.obs['biological_sample'].value_counts()

# Print the exact counts
print("Cell counts per biological sample:")
print(cell_counts)

# 2. Generate a bar plot (categorical equivalent of a histogram)
plt.figure(figsize=(15, 6)) # Make the figure wide so all 55 categories fit

# Plotting the counts
cell_counts.plot(kind='bar', color='skyblue', edgecolor='black')

# Formatting the chart
plt.title('Number of Cells per Sample', fontsize=16)
plt.xlabel('Biological Sample Category', fontsize=12)
plt.ylabel('Number of Cells', fontsize=12)

# Rotate x-axis labels to 90 degrees so the 55 labels don't overlap
plt.xticks(rotation=90, fontsize=8) 

# Ensure the labels aren't cut off when saving or displaying
plt.tight_layout() 
plt.show()

In [ ]:
len(cell_counts)

In [ ]:
import scanpy as sc

# 1. Hardcoded mapping dictionary from your file
sample_mapping_dict = {
    '1-1': 'D0_hPSC_rep1', '1-2': 'D0_hPSC_rep2', '1-3': 'D0_hPSC_rep3', 
    '1-4': 'D1_MPS_rep1', '1-5': 'D1_MPS_rep2', '1-6': 'D1_MPS_rep3', 
    '1-7': 'D1_MPS+TDI_rep1', '1-8': 'D1_MPS+TDI_rep2', '1-9': 'D1_MPS+TDI_rep3', 
    '1-10': 'D1_MPS+IAG_rep1', '1-11': 'D1_MPS+IAG_rep2', '1-12': 'D1_MPS+IAG_rep3', 
    '2-1': 'D1.33_LM_rep1', '2-2': 'D1.33_LM_rep2', '2-3': 'D1.33_LM_rep3', 
    '2-4': 'D1.33_LM+TDI_rep1', '2-5': 'D1.33_LM+TDI_rep2', '2-6': 'D1.33_LM+TDI_rep3', 
    '2-7': 'D1.33_LM+IAG_rep1', '2-8': 'D1.33_LM+IAG_rep2', '2-9': 'D1.33_LM+IAG_rep3', 
    '3-1': 'D1.67_LM_rep1', '3-2': 'D1.67_LM_rep2', '3-3': 'D1.67_LM_rep3', 
    '3-4': 'D1.67_LM+TDI_rep1', '3-5': 'D1.67_LM+TDI_rep2', '3-6': 'D1.67_LM+TDI_rep3', 
    '3-7': 'D1.67_LM+IAG_rep1', '3-8': 'D1.67_LM+IAG_rep2', '3-9': 'D1.67_LM+IAG_rep3', 
    '4-1': 'D2_LM_rep1', '4-2': 'D2_LM_rep2', '4-3': 'D2_LM_rep3', 
    '4-4': 'D2_LM+TDI_rep1', '4-5': 'D2_LM+TDI_rep2', '4-6': 'D2_LM+TDI_rep3', 
    '4-7': 'D2_LM+IAG_rep1', '4-8': 'D2_LM+IAG_rep2', '4-9': 'D2_LM+IAG_rep3', 
    '5-1': 'D3_AE_rep1', '5-2': 'D3_AE_rep2', '5-3': 'D3_AE_rep3', 
    '5-4': 'D3_AE+TDI_rep1', '5-5': 'D3_AE+TDI_rep2', '5-6': 'D3_AE+TDI_rep3', 
    '5-7': 'D3_AE+IAG_rep1', '5-8': 'D3_AE+IAG_rep2', '5-9': 'D3_AE+IAG_rep3', 
    '5-10': 'D2_LM+TDI->D3_AE+TDI_rep1', '5-11': 'D2_LM+TDI->D3_AE+TDI_rep2', 
    '5-12': 'D2_LM+TDI->D3_AE+TDI_rep3', '5-13': 'D2_LM+IAG->D3_AE+IAG_rep1', 
    '5-14': 'D2_LM+IAG->D3_AE+IAG_rep2', '5-15': 'D2_LM+IAG->D3_AE+IAG_rep3'
}

# 2. Ensure biological_sample is a string to match the dictionary keys
adata.obs['biological_sample'] = adata.obs['biological_sample'].astype(str).str.strip()

# 3. Map the biological_sample to the new sample_name column
adata.obs['sample_name'] = adata.obs['biological_sample'].map(sample_mapping_dict)

# 4. Convert the new column to a categorical variable (Scanpy best practice)
adata.obs['sample_name'] = adata.obs['sample_name'].astype('category')

# 5. Verify the new column
print(adata.obs[['biological_sample', 'sample_name']].head())

In [ ]:
import matplotlib.pyplot as plt

# 1. Count the number of cells in each category
cell_counts = adata.obs['sample_name'].value_counts()

# Print the exact counts
print("Cell counts per biological sample:")
print(cell_counts)

# 2. Generate a bar plot (categorical equivalent of a histogram)
plt.figure(figsize=(15, 6)) # Make the figure wide so all 55 categories fit

# Plotting the counts
cell_counts.plot(kind='bar', color='skyblue', edgecolor='black')

# Formatting the chart
plt.title('Number of Cells per Sample', fontsize=16)
plt.xlabel('Biological Sample Category', fontsize=12)
plt.ylabel('Number of Cells', fontsize=12)

# Rotate x-axis labels to 90 degrees so the 55 labels don't overlap
plt.xticks(rotation=90, fontsize=8) 

# Ensure the labels aren't cut off when saving or displaying
plt.tight_layout() 
plt.show()

In [ ]:
adata.shape

#### cNMF Projection

In [ ]:
adata.write_h5ad('/scratch/users/edtng/20260325_Hippo_timecourse/Hippo_timecourse_combined.h5ad')

In [ ]:
adata.var['gene']

In [ ]:
# %% [markdown]
# 
# # Project Hippo inhibitor time-course scRNA-seq onto perturb-seq cNMF programs
# 
# This notebook projects a new Hippo inhibitor time-course AnnData object onto the 50 cNMF gene programs discovered in the perturb-seq dataset. Program IDs are relabeled to zero-based `P0`–`P49` everywhere.
# 
# Reference matrix used:
# 
# ```text
# 030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50.gene_spectra_tpm.k_50.dt_0_2.txt
# ```
# 
# The notebook keeps the original `adata.X` unchanged. It computes a CPM/TPM-like working matrix from raw counts, projects cells onto fixed perturb-seq program spectra, then saves:
# 
# - projected `.h5ad`
# - projected program-score table
# - `.h5mu` with modalities `rna` and `perturbseq_cNMF`

# %%

from pathlib import Path
import re
import time
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from scipy.optimize import nnls

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import muon as mu
except Exception as e:
    mu = None
    print("WARNING: muon could not be imported; .h5mu saving will be skipped unless muon is installed.")
    print(e)

warnings.filterwarnings("ignore")

# -------------------------
# Input/output paths
# -------------------------
ADATA_PATH = Path("/scratch/users/edtng/20260325_Hippo_timecourse/Hippo_timecourse_combined.h5ad")

SPECTRA_TPM_PATH = Path(
    "/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result/"
    "030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50/"
    "030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50."
    "gene_spectra_tpm.k_50.dt_0_2.txt"
)

# Optional; used only to add labels if the file exists.
PROGRAM_META_PATH = Path(
    "/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/data/program_activity_long_v4.csv"
)

OUT_DIR = ADATA_PATH.parent / "perturbseq_cNMF_projection"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_H5AD = OUT_DIR / "Hippo_timecourse_combined.projected_perturbseq_cNMF_50_dt_0_2.zero_based.h5ad"
OUT_H5MU = OUT_DIR / "Hippo_timecourse_combined.projected_perturbseq_cNMF_50_dt_0_2.zero_based.h5mu"
OUT_SCORES_TSV = OUT_DIR / "Hippo_timecourse_combined.perturbseq_cNMF_50_dt_0_2.program_scores.zero_based.tsv.gz"

# -------------------------
# Projection settings
# -------------------------
TARGET_SUM = 1_000_000  # cNMF spectra_tpm is TPM/CPM-like, so use non-log normalized counts.

# Gene IDs in query AnnData.
# Use "var_names" if adata.var_names are gene symbols matching spectra columns.
# Otherwise set QUERY_GENE_ID_SOURCE = "column" and QUERY_GENE_COLUMN = "gene_name" or similar.
QUERY_GENE_ID_SOURCE = "column"  # "var_names" or "column"
QUERY_GENE_COLUMN = "gene"

# Counts source. If None, the notebook auto-detects a counts layer; if none exists, it uses adata.X.
COUNTS_LAYER = None
PREFERRED_COUNTS_LAYERS = ["counts", "raw_counts", "count", "raw"]

# Projection method:
# - "sklearn_fixed": fast fixed-H NMF-style projection; recommended default.
# - "fast_lstsq_clip": very fast approximate projection; clips negative least-squares coefficients to 0.
# - "nnls": exact scipy NNLS per cell; slow for large datasets.
PROJECTION_METHOD = "sklearn_fixed"
SKLEARN_MAX_ITER = 1000
SKLEARN_TOL = 1e-5

# Store obs columns for every program. This is convenient but adds 50 columns to adata.obs.
ADD_PROGRAM_COLUMNS_TO_OBS = True

# Keep raw usage and normalized usage. Normalized usage makes programs comparable within each cell.
NORMALIZE_USAGE_TO_SUM1 = True

print("ADATA_PATH:", ADATA_PATH)
print("SPECTRA_TPM_PATH:", SPECTRA_TPM_PATH)
print("OUT_DIR:", OUT_DIR)

# %%

# Load and inspect the new time-course AnnData
adata = sc.read_h5ad(ADATA_PATH)

print(adata)
print("\nobs columns:")
print(list(adata.obs.columns))
print("\nvar columns:")
print(list(adata.var.columns))
print("\nlayers:")
print(list(adata.layers.keys()))
print("\nobsm:")
print(list(adata.obsm.keys()))
print("\nuns keys:")
print(list(adata.uns.keys())[:50])

print("\nFirst obs rows:")
display(adata.obs.head())
print("\nFirst var rows:")
display(adata.var.head())

# %%

# Load reference perturb-seq cNMF spectra: programs x genes
spectra_df = pd.read_csv(SPECTRA_TPM_PATH, sep="\t", index_col=0)
spectra_df.index = spectra_df.index.astype(str)
spectra_df.columns = spectra_df.columns.astype(str)

# Make sure values are non-negative and numeric.
spectra_df = spectra_df.apply(pd.to_numeric, errors="coerce").fillna(0.0)
spectra_df[spectra_df < 0] = 0.0

# cNMF output files label programs as 1..K, but the downstream perturb-seq
# program metadata/mdata columns use 0-based IDs. Convert spectra row labels:
# source row "1" -> program_id 0 -> display name "P0"
# source row "50" -> program_id 49 -> display name "P49"
spectra_source_programs = spectra_df.index.astype(str).tolist()
spectra_program_int = [int(x) for x in spectra_source_programs]

if spectra_program_int == list(range(1, len(spectra_program_int) + 1)):
    program_ids = [x - 1 for x in spectra_program_int]
else:
    # Fallback: if a future spectra file is already zero-based, keep it.
    program_ids = spectra_program_int

program_names = [f"P{i}" for i in program_ids]

# Keep the spectra rows in their original order, but relabel the index to P0..P49.
spectra_df.index = program_names
reference_genes = spectra_df.columns.astype(str).tolist()

program_name_to_id = dict(zip(program_names, program_ids))
program_name_to_source_row = dict(zip(program_names, spectra_source_programs))

expected_zero_based_names = [f"P{i}" for i in range(len(program_names))]
if program_names != expected_zero_based_names:
    raise ValueError(
        "Program relabeling did not produce contiguous zero-based names. "
        f"Got first names {program_names[:10]} and last names {program_names[-5:]}"
    )
if "P50" in program_names:
    raise ValueError("P50 found after relabeling; expected P0..P49 only.")

print("spectra_df shape, programs x genes:", spectra_df.shape)
print("source programs from spectra:", spectra_source_programs[:10])
print("zero-based program names:", program_names[:10])
print("first genes:", reference_genes[:10])
print("row sum range:", float(spectra_df.sum(axis=1).min()), float(spectra_df.sum(axis=1).max()))

# Optional program labels/colors from metadata, if available.
program_meta = None
program_label_map = {p: p for p in program_names}

if PROGRAM_META_PATH.exists():
    program_meta = pd.read_csv(PROGRAM_META_PATH).drop_duplicates("program_name")
    if "program_label" in program_meta.columns:
        # program_activity_long_v4.csv uses 0-based program IDs.
        for _, row in program_meta.iterrows():
            if pd.isna(row["program_name"]):
                continue
            pid = int(row["program_name"])
            pname = f"P{pid}"
            label = str(row["program_label"])
            program_label_map[pname] = label
    print("Loaded program metadata:", PROGRAM_META_PATH)
else:
    print("Program metadata not found; proceeding without labels:", PROGRAM_META_PATH)

# %%

def choose_counts_layer(adata, requested_layer=None, preferred_layers=None):
    """Return a count matrix and a human-readable source name."""
    preferred_layers = preferred_layers or []

    if requested_layer is not None:
        if requested_layer not in adata.layers:
            raise KeyError(f"Requested COUNTS_LAYER={requested_layer!r} not found. Available layers: {list(adata.layers.keys())}")
        return adata.layers[requested_layer], f"layers[{requested_layer!r}]"

    for layer in preferred_layers:
        if layer in adata.layers:
            return adata.layers[layer], f"layers[{layer!r}]"

    # Fall back to adata.X. The user said this matrix is currently unnormalized.
    return adata.X, "adata.X"


def get_query_gene_ids(adata, source="var_names", column=None):
    if source == "var_names":
        return np.asarray(adata.var_names.astype(str))
    if source == "column":
        if column is None:
            raise ValueError("QUERY_GENE_COLUMN must be set when QUERY_GENE_ID_SOURCE='column'.")
        if column not in adata.var.columns:
            raise KeyError(f"QUERY_GENE_COLUMN={column!r} not found in adata.var. Available columns: {list(adata.var.columns)}")
        return np.asarray(adata.var[column].astype(str))
    raise ValueError("QUERY_GENE_ID_SOURCE must be 'var_names' or 'column'.")


def as_csr_counts_matrix(X):
    """Convert an AnnData matrix/layer to CSR sparse format for efficient gene aggregation."""
    if sparse.issparse(X):
        return X.tocsr()
    # scRNA-seq counts should usually be sparse. Dense conversion can be memory-heavy.
    print("WARNING: count matrix is dense; converting to sparse CSR. This may use substantial memory.")
    return sparse.csr_matrix(X)


def build_gene_aggregation_matrix(query_gene_ids, common_genes):
    """Build a sparse var x common_genes matrix that sums duplicate query genes if needed."""
    common_to_col = {g: j for j, g in enumerate(common_genes)}
    rows = []
    cols = []
    data = []
    matched_vars = 0

    for i, gene in enumerate(query_gene_ids):
        j = common_to_col.get(gene)
        if j is not None:
            rows.append(i)
            cols.append(j)
            data.append(1.0)
            matched_vars += 1

    G = sparse.coo_matrix(
        (np.asarray(data, dtype=np.float32), (np.asarray(rows), np.asarray(cols))),
        shape=(len(query_gene_ids), len(common_genes)),
    ).tocsr()

    return G, matched_vars


def sparse_row_sums(X):
    return np.asarray(X.sum(axis=1)).ravel()

# %%

# Extract raw counts, compute full-library-size scaling, then subset/aggregate to reference genes.
counts_matrix_raw, counts_source = choose_counts_layer(
    adata,
    requested_layer=COUNTS_LAYER,
    preferred_layers=PREFERRED_COUNTS_LAYERS,
)
counts_csr = as_csr_counts_matrix(counts_matrix_raw)

query_gene_ids = get_query_gene_ids(
    adata,
    source=QUERY_GENE_ID_SOURCE,
    column=QUERY_GENE_COLUMN,
)
query_gene_ids = np.asarray([str(x) for x in query_gene_ids])

query_gene_set = set(query_gene_ids)
common_genes = [g for g in reference_genes if g in query_gene_set]

print("Counts source:", counts_source)
print("Query matrix shape:", counts_csr.shape)
print("Reference genes:", len(reference_genes))
print("Query genes:", len(query_gene_ids))
print("Shared genes:", len(common_genes))

if len(common_genes) < 500:
    raise ValueError(
        f"Only {len(common_genes)} shared genes. Check whether adata.var_names are gene symbols. "
        "If gene symbols are in adata.var, set QUERY_GENE_ID_SOURCE='column' and QUERY_GENE_COLUMN accordingly."
    )

# Full-library cell sums should be computed over all query genes, not only the shared genes.
cell_sums = sparse_row_sums(counts_csr).astype(np.float64)
zero_cells = int(np.sum(cell_sums <= 0))
print("Cells with zero total counts:", zero_cells)
if zero_cells > 0:
    cell_sums[cell_sums <= 0] = np.nan

scale = TARGET_SUM / cell_sums
scale = np.nan_to_num(scale, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

# Aggregate duplicate gene symbols if present.
G, matched_vars = build_gene_aggregation_matrix(query_gene_ids, common_genes)
print("Matched query var rows, allowing duplicates:", matched_vars)

X_common_counts = counts_csr @ G
X_common_cpm = X_common_counts.multiply(scale[:, None]).tocsr().astype(np.float32)

print("X_common_cpm shape, cells x shared_genes:", X_common_cpm.shape)
print("X_common_cpm nnz:", X_common_cpm.nnz)
print("Mean nonzero value:", float(X_common_cpm.data.mean()) if X_common_cpm.nnz else 0.0)

# %%

# Align the reference spectra to the shared genes.
H = spectra_df.loc[program_names, common_genes].to_numpy(dtype=np.float32)  # programs x genes
H[H < 0] = 0

# Remove genes that have zero loading across all programs or zero expression across all cells.
h_nonzero = np.asarray(H.sum(axis=0)).ravel() > 0
x_nonzero = np.asarray(X_common_cpm.sum(axis=0)).ravel() > 0
keep_gene_mask = h_nonzero & x_nonzero

kept_genes = [g for g, keep in zip(common_genes, keep_gene_mask) if keep]
X_proj = X_common_cpm[:, keep_gene_mask].tocsr()
H_proj = H[:, keep_gene_mask]

print("Projection matrix X_proj, cells x genes:", X_proj.shape)
print("Reference matrix H_proj, programs x genes:", H_proj.shape)
print("Programs:", len(program_names))
print("Genes removed because zero in query or reference:", len(common_genes) - len(kept_genes))

if X_proj.shape[1] < 500:
    raise ValueError(f"Only {X_proj.shape[1]} usable genes after filtering. Check gene IDs/preprocessing.")

# %%

def initial_usage_lstsq_clip(X, H):
    """Fast nonnegative initialization by unconstrained least squares followed by clipping."""
    # X: cells x genes, H: programs x genes
    # Solve W H ~= X => W ~= X H.T (H H.T)^-1
    gram = H @ H.T
    gram_pinv = np.linalg.pinv(gram).astype(np.float32)
    XHt = X @ H.T
    W0 = np.asarray(XHt @ gram_pinv, dtype=np.float32)
    W0[W0 < 0] = 0
    return W0


def project_sklearn_fixed_H(X, H, max_iter=300, tol=1e-4):
    """Project cells onto fixed nonnegative spectra using sklearn's fixed-H NMF update."""
    try:
        from sklearn.decomposition import non_negative_factorization
    except Exception as e:
        raise ImportError("scikit-learn is not available; use PROJECTION_METHOD='fast_lstsq_clip' or 'nnls'.") from e

    n_cells = X.shape[0]
    n_programs = H.shape[0]
    print("Computing least-squares nonnegative initialization...")
    W0 = initial_usage_lstsq_clip(X, H)
    if not np.any(W0):
        W0 = np.full((n_cells, n_programs), 1e-6, dtype=np.float32)

    print("Running sklearn non_negative_factorization with fixed H...")
    W_fit, H_fit, n_iter = non_negative_factorization(
        X,
        W=W0,
        H=H.copy(),
        n_components=n_programs,
        init="custom",
        update_H=False,
        solver="cd",
        beta_loss="frobenius",
        max_iter=max_iter,
        tol=tol,
        random_state=0,
        verbose=1,
        shuffle=False,
    )
    print("sklearn iterations:", n_iter)
    W_fit = np.asarray(W_fit, dtype=np.float32)
    W_fit[W_fit < 0] = 0
    return W_fit


def project_nnls(X, H, report_every=1000):
    """Exact NNLS per cell. This can be slow for large datasets."""
    A = H.T.astype(np.float64)  # genes x programs
    n_cells = X.shape[0]
    n_programs = H.shape[0]
    W = np.zeros((n_cells, n_programs), dtype=np.float32)
    t0 = time.time()

    for i in range(n_cells):
        xi = X[i, :].toarray().ravel().astype(np.float64) if sparse.issparse(X) else np.asarray(X[i, :]).ravel().astype(np.float64)
        W[i, :], _ = nnls(A, xi)
        if (i + 1) % report_every == 0:
            elapsed = time.time() - t0
            print(f"NNLS projected {i + 1:,}/{n_cells:,} cells; elapsed {elapsed / 60:.1f} min")

    return W


t0 = time.time()

if PROJECTION_METHOD == "sklearn_fixed":
    usage_raw = project_sklearn_fixed_H(X_proj, H_proj, max_iter=SKLEARN_MAX_ITER, tol=SKLEARN_TOL)
elif PROJECTION_METHOD == "fast_lstsq_clip":
    usage_raw = initial_usage_lstsq_clip(X_proj, H_proj)
elif PROJECTION_METHOD == "nnls":
    usage_raw = project_nnls(X_proj, H_proj)
else:
    raise ValueError("PROJECTION_METHOD must be 'sklearn_fixed', 'fast_lstsq_clip', or 'nnls'.")

elapsed = time.time() - t0
print(f"Projection complete in {elapsed / 60:.2f} min")
print("usage_raw shape, cells x programs:", usage_raw.shape)
print("usage_raw min/max:", float(np.min(usage_raw)), float(np.max(usage_raw)))

# Normalize each cell's projected usage to sum to 1. This is usually easier to compare across cells.
usage_sum = usage_raw.sum(axis=1, keepdims=True).astype(np.float32)
usage_norm = np.divide(
    usage_raw,
    usage_sum,
    out=np.zeros_like(usage_raw, dtype=np.float32),
    where=usage_sum > 0,
)

print("usage_norm row sum range:", float(usage_norm.sum(axis=1).min()), float(usage_norm.sum(axis=1).max()))

# %%

# Create a zero-based program-score DataFrame for inspection and plotting.
program_score_df = pd.DataFrame(
    usage_norm if NORMALIZE_USAGE_TO_SUM1 else usage_raw,
    index=adata.obs_names.astype(str),
    columns=[str(p) for p in program_names],
)

print(program_score_df.shape)
display(program_score_df.iloc[:5, :10])

# Optional quick summary by time/day if such fields exist.
possible_time_cols = [c for c in ["day", "time", "timepoint", "treatment", "condition"] if c in adata.obs.columns]
print("Possible time/condition columns found:", possible_time_cols)

if "day" in adata.obs.columns:
    summary_by_day = program_score_df.join(adata.obs[["day"]]).groupby("day", observed=True).mean()
    display(summary_by_day.iloc[:, :10])

# %%

def sanitize_program_name(x):
    x = str(x)
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    if re.match(r"^[0-9]", x):
        x = "P" + x
    return x

# Attach scores to AnnData.
adata.obsm["X_perturbseq_cNMF_usage_raw"] = usage_raw.astype(np.float32)
adata.obsm["X_perturbseq_cNMF_usage"] = usage_norm.astype(np.float32)

if ADD_PROGRAM_COLUMNS_TO_OBS:
    for j, program in enumerate(program_names):
        col = f"perturbseq_cNMF_{sanitize_program_name(program)}_usage"
        adata.obs[col] = usage_norm[:, j] if NORMALIZE_USAGE_TO_SUM1 else usage_raw[:, j]

adata.uns["perturbseq_cNMF_projection"] = {
    "reference": "030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50",
    "k": 50,
    "density_threshold": "0.2",
    "spectra_tpm_path": str(SPECTRA_TPM_PATH),
    "query_adata_path": str(ADATA_PATH),
    "counts_source": counts_source,
    "target_sum": TARGET_SUM,
    "projection_method": PROJECTION_METHOD,
    "program_names": [str(p) for p in program_names],
    "program_column_names": [f"perturbseq_cNMF_{sanitize_program_name(p)}_usage" for p in program_names],
    "program_ids_zero_based": [int(program_name_to_id[p]) for p in program_names],
    "spectra_source_programs_one_based": [str(program_name_to_source_row[p]) for p in program_names],
    "program_labels": [str(program_label_map.get(str(p), str(p))) for p in program_names],
    "n_shared_genes_initial": int(len(common_genes)),
    "n_genes_used": int(len(kept_genes)),
    "genes_used": [str(g) for g in kept_genes],
    "usage_obsm_raw": "X_perturbseq_cNMF_usage_raw",
    "usage_obsm_normalized": "X_perturbseq_cNMF_usage",
}

print("Added obsm keys:", [k for k in adata.obsm.keys() if "perturbseq" in k])
print("Added obs score columns:", sum(c.startswith("perturbseq_cNMF_") for c in adata.obs.columns))

# %%

# Save projected score table and projected AnnData.
program_score_df.to_csv(OUT_SCORES_TSV, sep="\t", compression="gzip")
print("Saved score table:", OUT_SCORES_TSV)

adata.write_h5ad(OUT_H5AD, compression="gzip")
print("Saved projected AnnData:", OUT_H5AD)

# %%

# Save a MuData file with two modalities:
#   - rna: original AnnData plus projection in obsm/obs
#   - perturbseq_cNMF: cells x projected perturb-seq programs
if mu is None:
    print("muon is not installed/importable; skipping .h5mu save.")
else:
    program_var = pd.DataFrame(index=pd.Index([str(p) for p in program_names], name="program"))
    program_var["program_name"] = [str(p) for p in program_names]
    program_var["program_id"] = [int(program_name_to_id[p]) for p in program_names]
    program_var["program_id_zero_based"] = [int(program_name_to_id[p]) for p in program_names]
    program_var["spectra_source_program_one_based"] = [str(program_name_to_source_row[p]) for p in program_names]
    program_var["program_label"] = [str(program_label_map.get(str(p), str(p))) for p in program_names]
    program_var["reference"] = "perturbseq_cNMF_50_dt_0_2"

    # Store minimal obs in the program modality; MuData will align cells by obs_names.
    program_obs = pd.DataFrame(index=adata.obs_names.copy())
    for col in ["day", "time", "timepoint", "treatment", "condition"]:
        if col in adata.obs.columns:
            program_obs[col] = adata.obs[col].values

    prog_adata = sc.AnnData(
        X=usage_norm.astype(np.float32),
        obs=program_obs,
        var=program_var,
    )
    prog_adata.layers["usage_raw"] = usage_raw.astype(np.float32)
    prog_adata.layers["usage_sum_to_1"] = usage_norm.astype(np.float32)
    prog_adata.uns["projection"] = adata.uns["perturbseq_cNMF_projection"]

    # Carry over UMAP coordinates if present.
    for key in ["X_umap", "X_pca"]:
        if key in adata.obsm:
            prog_adata.obsm[key] = adata.obsm[key].copy()

    mdata = mu.MuData({
        "rna": adata,
        "perturbseq_cNMF": prog_adata,
    })
    mdata.uns["description"] = (
        "Hippo inhibitor time-course scRNA-seq projected onto perturb-seq cNMF programs "
        "using gene_spectra_tpm.k_50.dt_0_2."
    )

    mdata.write_h5mu(OUT_H5MU)
    print("Saved projected MuData:", OUT_H5MU)

# %%

# Reload sanity checks.
adata_check = sc.read_h5ad(OUT_H5AD, backed="r")
print(adata_check)
print("Projection obsm keys:", [k for k in adata_check.obsm.keys() if "perturbseq" in k])
score_cols_check = [
    c for c in adata_check.obs.columns
    if c.startswith("perturbseq_cNMF_P") and c.endswith("_usage")
]
print("obs score columns:", len(score_cols_check))
print("first obs score columns:", score_cols_check[:5])
print("last obs score columns:", score_cols_check[-5:])
if any("P50" in c for c in score_cols_check):
    raise ValueError("Found P50 in obs score columns; expected zero-based P0..P49 only.")
adata_check.file.close()

if OUT_H5MU.exists() and mu is not None:
    mdata_check = mu.read_h5mu(OUT_H5MU, backed="r")
    print(mdata_check)
    print("modalities:", list(mdata_check.mod.keys()))
    print("program modality shape:", mdata_check.mod["perturbseq_cNMF"].shape)
    print("program modality first vars:", mdata_check.mod["perturbseq_cNMF"].var_names[:5].tolist())
    print("program modality last vars:", mdata_check.mod["perturbseq_cNMF"].var_names[-5:].tolist())
    if "P50" in set(mdata_check.mod["perturbseq_cNMF"].var_names.astype(str)):
        raise ValueError("Found P50 in MuData program modality; expected zero-based P0..P49 only.")
    mdata_check.file.close()

# %%

# Optional quick plot of a projected program over an existing UMAP.
# Set PROGRAM_TO_PLOT to a program name from program_names, e.g. program_names[0].
import matplotlib.pyplot as plt

PROGRAM_TO_PLOT = program_names[0]

if "X_umap" not in adata.obsm:
    print("No adata.obsm['X_umap'] found; skipping UMAP plot.")
else:
    j = program_names.index(PROGRAM_TO_PLOT)
    vals = usage_norm[:, j]
    coords = adata.obsm["X_umap"]

    fig, ax = plt.subplots(figsize=(5, 5), dpi=150)
    sca = ax.scatter(coords[:, 0], coords[:, 1], c=vals, s=1, linewidths=0, rasterized=True)
    ax.set_title(f"Perturb-seq cNMF program {PROGRAM_TO_PLOT}")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    fig.colorbar(sca, ax=ax, label="Projected usage")
    plt.show()



#### Verify that projection ran correctly

In [ ]:
# %% [markdown]
# # Zero-based sanity checks for Hippo time-course projection
#
# This notebook checks the saved projection after relabeling perturb-seq cNMF
# programs to zero-based names: P0, P1, ..., P49.
#
# It does not recompute the projection.

# %%
from pathlib import Path
import re

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

try:
    import muon as mu
except Exception:
    mu = None

# -------------------------
# Paths
# -------------------------
OUT_DIR = Path("/scratch/users/edtng/20260325_Hippo_timecourse/perturbseq_cNMF_projection")

H5AD_PATH = OUT_DIR / "Hippo_timecourse_combined.projected_perturbseq_cNMF_50_dt_0_2.h5ad"
H5MU_PATH = OUT_DIR / "Hippo_timecourse_combined.projected_perturbseq_cNMF_50_dt_0_2.h5mu"
SCORE_TSV = OUT_DIR / "Hippo_timecourse_combined.perturbseq_cNMF_50_dt_0_2.program_scores.tsv.gz"

OLD_H5AD_PATH = OUT_DIR / "Hippo_timecourse_combined.projected_perturbseq_cNMF_50_dt_0_2.h5ad"

SPECTRA_TPM_PATH = Path(
    "/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result/"
    "030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50/"
    "030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50."
    "gene_spectra_tpm.k_50.dt_0_2.txt"
)

print("H5AD_PATH:", H5AD_PATH)
print("H5MU_PATH:", H5MU_PATH)
print("SCORE_TSV:", SCORE_TSV)
print("SPECTRA_TPM_PATH:", SPECTRA_TPM_PATH)

# %%
# Helper functions

def p_to_int(program):
    """Convert 'P38', 38, '38' to integer 38."""
    s = str(program)
    s = re.sub(r"^perturbseq_cNMF_", "", s)
    s = re.sub(r"_usage$", "", s)
    s = s.replace("P", "")
    return int(s)


def expected_zero_based_program_names(n_programs):
    return [f"P{i}" for i in range(n_programs)]


def obs_score_cols_zero_based(adata):
    cols = [
        c for c in adata.obs.columns
        if c.startswith("perturbseq_cNMF_P") and c.endswith("_usage")
    ]
    return sorted(cols, key=p_to_int)


def get_program_names_from_saved_adata(adata, n_programs):
    """Prefer saved uns program names. Fallback to obs score columns. Final fallback P0..Pn-1."""
    uns = adata.uns.get("perturbseq_cNMF_projection", {})

    if "program_names" in uns:
        names = [str(x) for x in uns["program_names"]]
        if len(names) == n_programs:
            return names

    cols = obs_score_cols_zero_based(adata)
    if len(cols) == n_programs:
        return [re.sub(r"^perturbseq_cNMF_", "", c).replace("_usage", "") for c in cols]

    return expected_zero_based_program_names(n_programs)


def load_zero_based_spectra(path):
    """Load spectra and relabel rows to P0..P49 if the source rows are 1..50."""
    spectra = pd.read_csv(path, sep="\t", index_col=0)
    spectra.index = spectra.index.astype(str)
    spectra.columns = spectra.columns.astype(str)
    spectra = spectra.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    spectra[spectra < 0] = 0.0

    source = spectra.index.astype(str).tolist()
    try:
        source_int = [int(x) for x in source]
    except Exception:
        source_int = None

    if source_int == list(range(1, len(source_int) + 1)):
        zero_names = [f"P{i - 1}" for i in source_int]
        mapping = pd.DataFrame({
            "spectra_source_program": source,
            "zero_based_program": zero_names,
            "old_one_based_label": [f"P{i}" for i in source_int],
        })
        spectra.index = zero_names
    elif source_int == list(range(0, len(source_int))):
        zero_names = [f"P{i}" for i in source_int]
        mapping = pd.DataFrame({
            "spectra_source_program": source,
            "zero_based_program": zero_names,
            "old_one_based_label": [f"P{i + 1}" for i in source_int],
        })
        spectra.index = zero_names
    else:
        # Fallback: preserve source labels but force P prefix for numeric names.
        zero_names = [x if x.startswith("P") else f"P{x}" for x in source]
        mapping = pd.DataFrame({
            "spectra_source_program": source,
            "zero_based_program": zero_names,
            "old_one_based_label": pd.NA,
        })
        spectra.index = zero_names

    return spectra, mapping


def matrix_qc(mat, name):
    print(f"\n{name}")
    print("shape:", mat.shape)
    print("dtype:", mat.dtype)
    print("NaN count:", int(np.isnan(mat).sum()))
    print("Inf count:", int(np.isinf(mat).sum()))
    print("min:", float(np.nanmin(mat)))
    print("max:", float(np.nanmax(mat)))
    print("negative count:", int((mat < -1e-8).sum()))
    print("exact zero fraction:", float(np.mean(mat == 0)))

# %%
# Load saved zero-based projection.
# Use backed='r' so adata.X is not loaded into memory.

if not H5AD_PATH.exists():
    raise FileNotFoundError(
        f"Missing zero-based h5ad: {H5AD_PATH}\n"
        "Use the relabel-only notebook/script first, or change H5AD_PATH."
    )

adata = sc.read_h5ad(H5AD_PATH, backed="r")

usage_norm = np.asarray(adata.obsm["X_perturbseq_cNMF_usage"])
usage_raw = np.asarray(adata.obsm["X_perturbseq_cNMF_usage_raw"])

n_cells, n_programs = usage_norm.shape
program_names = get_program_names_from_saved_adata(adata, n_programs)
expected_programs = expected_zero_based_program_names(n_programs)

print(adata)
print("usage_norm shape:", usage_norm.shape)
print("usage_raw shape:", usage_raw.shape)
print("program_names first/last:", program_names[:5], program_names[-5:])
print("expected first/last:", expected_programs[:5], expected_programs[-5:])

assert usage_norm.shape == usage_raw.shape
assert n_programs == 50

if program_names != expected_programs:
    raise ValueError(
        "Program names are not zero-based P0..P49.\n"
        f"Observed first/last: {program_names[:5]} ... {program_names[-5:]}"
    )

print("Zero-based program names OK.")

# %%
# Check obs score columns match zero-based names and match obsm values.

score_cols = obs_score_cols_zero_based(adata)
print("Number of obs score columns:", len(score_cols))
print("first obs score columns:", score_cols[:5])
print("last obs score columns:", score_cols[-5:])

expected_cols = [f"perturbseq_cNMF_{p}_usage" for p in expected_programs]
missing = sorted(set(expected_cols) - set(score_cols), key=lambda x: p_to_int(x))
extra = sorted(set(score_cols) - set(expected_cols), key=lambda x: p_to_int(x))

print("missing expected cols:", missing[:10], "n=", len(missing))
print("extra score cols:", extra[:10], "n=", len(extra))

assert not missing
assert not extra

obs_scores = adata.obs[expected_cols].to_numpy(dtype=np.float32)
print("Max abs difference obs score columns vs obsm usage_norm:", float(np.max(np.abs(obs_scores - usage_norm))))

# %%
# Basic numerical QC.

matrix_qc(usage_raw, "usage_raw")
matrix_qc(usage_norm, "usage_norm")

row_sums_raw = usage_raw.sum(axis=1)
row_sums_norm = usage_norm.sum(axis=1)

print("\nRaw row-sum summary:")
display(pd.Series(row_sums_raw).describe())

print("\nNormalized row-sum summary:")
display(pd.Series(row_sums_norm).describe())

print("Max abs deviation from normalized row sum = 1:", float(np.max(np.abs(row_sums_norm - 1))))

# %%
# Check score TSV, if available.

if SCORE_TSV.exists():
    score_df = pd.read_csv(SCORE_TSV, sep="\t", index_col=0)
    print("score_df shape:", score_df.shape)
    print("score_df first/last columns:", score_df.columns[:5].tolist(), score_df.columns[-5:].tolist())
    print("cell order matches AnnData:", np.array_equal(score_df.index.astype(str), adata.obs_names.astype(str)))
    assert score_df.columns.tolist() == expected_programs
    tsv_values = score_df.to_numpy(dtype=np.float32)
    print("Max abs difference TSV vs obsm usage_norm:", float(np.max(np.abs(tsv_values - usage_norm))))
else:
    print("No score TSV found:", SCORE_TSV)

# %%
# Check h5mu program modality, if available.

if H5MU_PATH.exists() and mu is not None:
    mdata = mu.read_h5mu(H5MU_PATH, backed="r")
    print(mdata)
    print("modalities:", list(mdata.mod.keys()))

    prog = mdata.mod["perturbseq_cNMF"]
    print("program modality shape:", prog.shape)
    print("program var_names first/last:", prog.var_names[:5].tolist(), prog.var_names[-5:].tolist())

    assert prog.n_obs == n_cells
    assert prog.n_vars == n_programs
    assert prog.var_names.tolist() == expected_programs

    prog_X = prog.X
    if sparse.issparse(prog_X):
        prog_X = prog_X.toarray()
    else:
        prog_X = np.asarray(prog_X)

    print("Max abs difference h5mu program X vs h5ad usage_norm:", float(np.max(np.abs(prog_X - usage_norm))))
    mdata.file.close()
else:
    print("Skipping h5mu check. Missing file or muon unavailable.")

# %%
# If old one-based file exists, confirm relabeling was only a label shift.
# Expected: old matrix column j equals new matrix column j, but labels changed
# old P1 -> new P0, old P2 -> new P1, ..., old P50 -> new P49.

if OLD_H5AD_PATH.exists():
    old = sc.read_h5ad(OLD_H5AD_PATH, backed="r")
    old_usage = np.asarray(old.obsm["X_perturbseq_cNMF_usage"])

    print("old usage shape:", old_usage.shape)
    print("new usage shape:", usage_norm.shape)
    print("max abs diff old usage vs new usage:", float(np.max(np.abs(old_usage - usage_norm))))

    mapping = pd.DataFrame({
        "matrix_column": np.arange(n_programs),
        "old_one_based_label": [f"P{i}" for i in range(1, n_programs + 1)],
        "new_zero_based_label": [f"P{i}" for i in range(n_programs)],
    })
    display(mapping.head(10))
    display(mapping.tail(10))
    old.file.close()
else:
    print("Old one-based H5AD not found; skipping old/new comparison:", OLD_H5AD_PATH)

# %%
# Program-level distribution summary with zero-based names.

program_summary = pd.DataFrame({
    "program": program_names,
    "mean": usage_norm.mean(axis=0),
    "median": np.median(usage_norm, axis=0),
    "p90": np.percentile(usage_norm, 90, axis=0),
    "p95": np.percentile(usage_norm, 95, axis=0),
    "p99": np.percentile(usage_norm, 99, axis=0),
    "max": usage_norm.max(axis=0),
    "frac_gt_0.01": (usage_norm > 0.01).mean(axis=0),
    "frac_gt_0.05": (usage_norm > 0.05).mean(axis=0),
    "frac_gt_0.10": (usage_norm > 0.10).mean(axis=0),
})
program_summary["old_one_based_label"] = program_summary["program"].map(lambda p: f"P{p_to_int(p) + 1}")
program_summary = program_summary.sort_values("mean", ascending=False)

display(program_summary)

# %%
# Dominant-program distribution, zero-based.

dominant_idx = np.argmax(usage_norm, axis=1)
dominant_program = np.array(program_names, dtype=object)[dominant_idx]
dominant_score = usage_norm.max(axis=1)

entropy = -np.sum(
    np.where(usage_norm > 0, usage_norm * np.log2(usage_norm), 0),
    axis=1,
)

dominant_df = pd.DataFrame({
    "dominant_program": dominant_program,
    "dominant_score": dominant_score,
    "usage_entropy": entropy,
}, index=adata.obs_names.astype(str))

print("Dominant score summary:")
display(dominant_df["dominant_score"].describe())

print("Usage entropy summary:")
display(dominant_df["usage_entropy"].describe())

program_counts = (
    dominant_df["dominant_program"]
    .value_counts()
    .rename_axis("program")
    .reset_index(name="n_cells")
)
program_counts["frac_cells"] = program_counts["n_cells"] / n_cells
program_counts["old_one_based_label"] = program_counts["program"].map(lambda p: f"P{p_to_int(p) + 1}")
display(program_counts)

plt.figure(figsize=(5, 3))
plt.hist(dominant_score, bins=100)
plt.xlabel("Dominant normalized program score")
plt.ylabel("Number of cells")
plt.title("Dominant program score per cell")
plt.tight_layout()
plt.show()

# %%
# QC correlations using BOTH raw and normalized program scores.
# This matters because usage_norm is compositional: every cell sums to 1.

candidate_qc_cols = [
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
    "pct_counts_ribo",
    "guide_umi_counts",
    "guides_per_cell",
]
available_qc_cols = [c for c in candidate_qc_cols if c in adata.obs.columns]
print("Available QC columns:", available_qc_cols)


def qc_corr_table(score_mat, score_type, use_spearman=False):
    records = []
    for qc_col in available_qc_cols:
        qc = pd.to_numeric(adata.obs[qc_col], errors="coerce").to_numpy()
        valid_qc = np.isfinite(qc)

        for j, program in enumerate(program_names):
            y = score_mat[:, j]
            valid = valid_qc & np.isfinite(y)
            if valid.sum() <= 10:
                pearson = np.nan
                spearman = np.nan
            else:
                pearson = np.corrcoef(qc[valid], y[valid])[0, 1]
                if use_spearman:
                    spearman = spearmanr(qc[valid], y[valid], nan_policy="omit").correlation
                else:
                    spearman = np.nan

            records.append({
                "score_type": score_type,
                "qc_col": qc_col,
                "program": program,
                "old_one_based_label": f"P{p_to_int(program) + 1}",
                "pearson_corr": pearson,
                "spearman_corr": spearman,
            })

    out = pd.DataFrame(records)
    out["abs_pearson"] = out["pearson_corr"].abs()
    out["abs_spearman"] = out["spearman_corr"].abs()
    return out

qc_raw = qc_corr_table(usage_raw, "raw", use_spearman=False)
qc_norm = qc_corr_table(usage_norm, "normalized", use_spearman=False)
qc_both = pd.concat([qc_raw, qc_norm], ignore_index=True)

# Put normalized first because those are likely what you will use downstream.
display(qc_both.sort_values(["abs_pearson", "score_type"], ascending=[False, True]).head(50))

# %%
# Mito/ribo/rRNA loading fractions in the reference spectra, using zero-based program names.
# This helps distinguish true mitochondrial/ribosomal programs from compositional artifacts.

spectra_zb, spectra_mapping = load_zero_based_spectra(SPECTRA_TPM_PATH)
print("spectra_zb shape:", spectra_zb.shape)
display(spectra_mapping.head(10))

assert spectra_zb.index.tolist() == expected_programs

genes = pd.Index(spectra_zb.columns.astype(str))
mt_mask = genes.str.startswith("MT-")
ribo_mask = genes.str.match(r"^(RPL|RPS|MRPL|MRPS)", na=False)
rRNA_mask = genes.str.contains("rRNA", case=False, regex=False)

spectra_qc = []
for program in expected_programs:
    row = spectra_zb.loc[program]
    denom = float(row.sum())
    if denom <= 0:
        denom = np.nan
    top_genes = row.sort_values(ascending=False).head(25).index.tolist()
    spectra_qc.append({
        "program": program,
        "old_one_based_label": f"P{p_to_int(program) + 1}",
        "mt_loading_fraction": float(row.loc[mt_mask].sum() / denom),
        "ribo_loading_fraction": float(row.loc[ribo_mask].sum() / denom),
        "rRNA_loading_fraction": float(row.loc[rRNA_mask].sum() / denom),
        "top_25_genes": ", ".join(top_genes),
    })

spectra_qc = pd.DataFrame(spectra_qc)

display(
    spectra_qc.sort_values(
        ["mt_loading_fraction", "ribo_loading_fraction", "rRNA_loading_fraction"],
        ascending=False,
    ).head(20)
)

# %%
# Merge normalized QC correlations with spectra QC annotations.
# Interpretation:
# - high pct_counts_mt correlation + high mt_loading_fraction => likely mitochondrial/QC-like program.
# - high pct_counts_ribo correlation + high ribo/rRNA loading => likely ribosomal/rRNA-like program.
# - high correlation but low loading => could be compositional, cell-state, or sample confounding.

norm_qc = qc_norm.copy()
merged_qc = norm_qc.merge(spectra_qc, on=["program", "old_one_based_label"], how="left")
merged_qc["abs_pearson"] = merged_qc["pearson_corr"].abs()

interesting_qc = merged_qc[
    merged_qc["qc_col"].isin(["pct_counts_mt", "pct_counts_ribo", "total_counts", "n_genes_by_counts"])
].sort_values("abs_pearson", ascending=False)

display(interesting_qc.head(50))

# %%
# Plot strongest mito/ribo-associated programs using zero-based labels.

plot_rows = interesting_qc[
    interesting_qc["qc_col"].isin(["pct_counts_mt", "pct_counts_ribo"])
].head(8)

rng = np.random.default_rng(0)
sample_idx = rng.choice(n_cells, size=min(30000, n_cells), replace=False)

for _, row in plot_rows.iterrows():
    qc_col = row["qc_col"]
    program = row["program"]
    j = p_to_int(program)

    x = pd.to_numeric(adata.obs[qc_col], errors="coerce").to_numpy()[sample_idx]
    y = usage_norm[sample_idx, j]

    plt.figure(figsize=(4, 3))
    plt.scatter(x, y, s=1, alpha=0.25)
    plt.xlabel(qc_col)
    plt.ylabel(f"{program} normalized usage")
    plt.title(f"{program} vs {qc_col} | old label {row['old_one_based_label']}")
    plt.tight_layout()
    plt.show()

# %%
# Sample/timepoint/group-level summary.
# The projection notebook did not auto-detect a day/time column, so inspect sample fields manually.

candidate_group_cols = [
    "sample_name",
    "biological_sample",
    "sample_id",
    "pool",
    "round1",
    "plate",
    "sublibID",
    "well",
]

for col in candidate_group_cols:
    if col in adata.obs.columns:
        print("\n" + col)
        print(adata.obs[col].value_counts(dropna=False).head(50))

# %%
# Choose the column encoding time/treatment and summarize program usage by group.
# Edit GROUP_COL after inspecting the previous output.

GROUP_COL = "sample_name"  # change if biological_sample or sample_id is better

if GROUP_COL in adata.obs.columns:
    score_df = pd.DataFrame(usage_norm, index=adata.obs_names.astype(str), columns=program_names)
    score_df[GROUP_COL] = adata.obs[GROUP_COL].astype(str).to_numpy()

    group_mean = score_df.groupby(GROUP_COL, observed=True)[program_names].mean()
    group_n = score_df.groupby(GROUP_COL, observed=True).size().rename("n_cells")

    display(group_n.to_frame())
    display(group_mean)

    # Z-score each program across groups for display only.
    mat_z = (group_mean - group_mean.mean(axis=0)) / group_mean.std(axis=0).replace(0, np.nan)

    plt.figure(figsize=(12, max(3, 0.25 * mat_z.shape[0])))
    plt.imshow(mat_z.to_numpy(), aspect="auto")
    plt.colorbar(label="Group mean usage, z-scored per program")
    plt.yticks(range(mat_z.shape[0]), mat_z.index)
    plt.xticks(range(mat_z.shape[1]), mat_z.columns, rotation=90)
    plt.title(f"Projected perturb-seq cNMF usage by {GROUP_COL}")
    plt.tight_layout()
    plt.show()

    dynamic = pd.DataFrame({
        "program": program_names,
        "old_one_based_label": [f"P{p_to_int(p) + 1}" for p in program_names],
        "min_group_mean": group_mean.min(axis=0).values,
        "max_group_mean": group_mean.max(axis=0).values,
    })
    dynamic["range"] = dynamic["max_group_mean"] - dynamic["min_group_mean"]
    dynamic = dynamic.sort_values("range", ascending=False)
    display(dynamic.head(30))
else:
    print(f"GROUP_COL={GROUP_COL!r} not found in adata.obs")

# %%
# Print top genes for QC-associated or dynamic programs.

PROGRAMS_TO_INSPECT = sorted(set(
    interesting_qc.head(12)["program"].tolist()
    + (dynamic.head(12)["program"].tolist() if "dynamic" in globals() else [])
), key=p_to_int)

for program in PROGRAMS_TO_INSPECT:
    row = spectra_zb.loc[program]
    top = row.sort_values(ascending=False).head(30).index.tolist()
    old_label = f"P{p_to_int(program) + 1}"
    print(f"\n{program}  old_one_based_label={old_label}")
    print(", ".join(top))

# %%
# Close backed AnnData handle.
adata.file.close()
print("Done.")


#### projected cNMF x sample_name

In [ ]:
# %% [markdown]
# # Projected perturb-seq cNMF program heatmap by sample_name
#
# This notebook/script loads the saved zero-based projected AnnData/MuData,
# applies condition-specific mitochondrial/ribosomal QC filters, and remakes a
# projected cNMF program x sample_name heatmap with all program names shown.

# %%
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt

try:
    import muon as mu
except ImportError:
    mu = None

# Edit paths here if needed.
OUT_DIR = Path("/scratch/users/edtng/20260325_Hippo_timecourse/perturbseq_cNMF_projection")
H5AD_PATH = OUT_DIR / "Hippo_timecourse_combined.projected_perturbseq_cNMF_50_dt_0_2.h5ad"
H5MU_PATH = OUT_DIR / "Hippo_timecourse_combined.projected_perturbseq_cNMF_50_dt_0_2.h5mu"
PROGRAM_META_PATH = Path("/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/data/program_activity_long_v4.csv")

PLOT_OUT_DIR = OUT_DIR / "heatmaps_after_qc_filter"
PLOT_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Use normalized projected usage by default. Change to raw if needed.
SCORE_KEY = "X_perturbseq_cNMF_usage"       # row-normalized cell x program usage
# SCORE_KEY = "X_perturbseq_cNMF_usage_raw" # unnormalized projected score

SAMPLE_COL = "sample_name"
CONDITION_COL = "condition"
SHOW_PROGRAM_LABELS = True
MAX_PROGRAM_LABEL_CHARS = 90

# Heatmap scaling:
# - "zscore_by_program": highlights relative changes across samples per program.
# - "mean": plots mean projected usage directly.
HEATMAP_SCALE = "zscore_by_program"
# HEATMAP_SCALE = "mean"

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",  # editable text in Illustrator
})


def show_df(df, n=20):
    """Display a dataframe in notebooks, or print in plain Python."""
    try:
        display(df)
    except NameError:
        print(df.head(n).to_string())

print("H5AD:", H5AD_PATH)
print("H5MU:", H5MU_PATH)
print("Output:", PLOT_OUT_DIR)

# %% [markdown]
# ## Load saved projected AnnData/MuData
#
# Load the AnnData in backed mode so the large gene-expression matrix is not read into memory.
# The projected cNMF matrix is only cells x 50 and will be loaded separately from `obsm`.

# %%
if not H5AD_PATH.exists():
    raise FileNotFoundError(f"Missing projected AnnData: {H5AD_PATH}")

adata = sc.read_h5ad(H5AD_PATH, backed="r")
print(adata)
print("obs columns:", list(adata.obs.columns))
print("obsm keys:", list(adata.obsm.keys()))

mdata = None
if mu is not None and H5MU_PATH.exists():
    try:
        mdata = mu.read_h5mu(H5MU_PATH, backed="r")
        print("\nLoaded MuData:")
        print(mdata)
        print("modalities:", list(mdata.mod.keys()))
    except Exception as e:
        print("Could not load MuData; continuing with AnnData only.")
        print(type(e).__name__, e)
elif mu is None:
    print("muon is not installed; continuing with AnnData only.")
else:
    print("MuData file not found; continuing with AnnData only:", H5MU_PATH)

# %% [markdown]
# ## Define QC cutoffs
#
# Cells are removed if either:
#
# ```text
# pct_counts_mt >= condition-specific mt cutoff
# OR
# pct_counts_ribo <= condition-specific ribo cutoff
# ```

# %%
cond_pct_ribo_cutofs = {
    'D0_hPSC': 2.4,
    'D1_MPS': 2.3,
    'D1_MPS+TDI': 2.5,
    'D1_MPS+IAG': 2.2,
    'D1.33_LM': 1.9,
    'D1.33_LM+TDI': 2.1,
    'D1.33_LM+IAG': 2.2,
    'D1.67_LM': 2.0,
    'D1.67_LM+TDI': 2.2,
    'D1.67_LM+IAG': 1.9,
    'D2_LM': 2.0,
    'D2_LM+TDI': 2.4,
    'D2_LM+IAG': 1.9,
    'D3_AE': 1.0,
    'D3_AE+TDI': 2.0,
    'D3_AE+IAG': 1.0,
    'D2_LM+TDI->D3_AE+TDI': 2.5,
    'D2_LM+IAG->D3_AE+IAG': 1.0,
}

cond_pct_mt_cutofs = {
    'D0_hPSC': 5.5,
    'D1_MPS': 3.0,
    'D1_MPS+TDI': 3.0,
    'D1_MPS+IAG': 4,
    'D1.33_LM': 3.5,
    'D1.33_LM+TDI': 3.75,
    'D1.33_LM+IAG': 5,
    'D1.67_LM': 4.0,
    'D1.67_LM+TDI': 5.5,
    'D1.67_LM+IAG': 3.5,
    'D2_LM': 6.5,
    'D2_LM+TDI': 8,
    'D2_LM+IAG': 3.5,
    'D3_AE': 3.25,
    'D3_AE+TDI': 7.0,
    'D3_AE+IAG': 3.75,
    'D2_LM+TDI->D3_AE+TDI': 7,
    'D2_LM+IAG->D3_AE+IAG': 3.75,
}

condition_order = list(cond_pct_mt_cutofs.keys())
assert set(cond_pct_mt_cutofs) == set(cond_pct_ribo_cutofs), "MT and ribo cutoff dictionaries have different condition keys."
print("Conditions with cutoffs:", len(condition_order))

# %% [markdown]
# ## Ensure/derive condition labels
#
# The filter uses `adata.obs[condition]`. If the projected file does not contain a `condition`
# column, this cell tries to infer it from `sample_name` by exact match or substring match.

# %%
obs = adata.obs.copy()

if SAMPLE_COL not in obs.columns:
    raise KeyError(f"SAMPLE_COL={SAMPLE_COL!r} is not present in adata.obs. Available columns: {list(obs.columns)}")


def infer_condition_from_sample(value, valid_conditions):
    """Infer condition from a sample string, preferring exact then longest substring match."""
    s = str(value)
    if s in valid_conditions:
        return s
    for cond in sorted(valid_conditions, key=len, reverse=True):
        if cond in s:
            return cond
    return np.nan

if CONDITION_COL not in obs.columns:
    inferred = obs[SAMPLE_COL].map(lambda x: infer_condition_from_sample(x, condition_order))
    frac_mapped = inferred.notna().mean()
    print(f"{CONDITION_COL!r} not found in obs. Inferred condition from {SAMPLE_COL!r}: {frac_mapped:.3%} mapped.")
    if frac_mapped < 1.0:
        unmapped = obs.loc[inferred.isna(), SAMPLE_COL].astype(str).value_counts().head(50)
        print("Unmapped sample_name values:")
        print(unmapped.to_string())
        raise ValueError(
            "Could not infer condition for all cells. Add adata.obs['condition'] or adjust infer_condition_from_sample()."
        )
    obs[CONDITION_COL] = inferred.astype(str)
else:
    obs[CONDITION_COL] = obs[CONDITION_COL].astype(str)

print("Condition counts:")
show_df(obs[CONDITION_COL].value_counts(dropna=False).rename_axis("condition").reset_index(name="n_cells"), n=100)

# %% [markdown]
# ## Apply condition-specific ribo/mito filters

# %%
required_cols = [CONDITION_COL, SAMPLE_COL, "pct_counts_mt", "pct_counts_ribo"]
missing_cols = [c for c in required_cols if c not in obs.columns]
if missing_cols:
    raise KeyError(f"Missing required columns in adata.obs: {missing_cols}")

observed_conditions = set(obs[CONDITION_COL].astype(str).unique())
missing_cutoffs = sorted(observed_conditions - set(condition_order))
if missing_cutoffs:
    raise KeyError(
        "These observed conditions do not have cutoffs: " + ", ".join(missing_cutoffs)
    )

mt = pd.to_numeric(obs["pct_counts_mt"], errors="coerce")
ribo = pd.to_numeric(obs["pct_counts_ribo"], errors="coerce")
if mt.isna().any() or ribo.isna().any():
    raise ValueError("pct_counts_mt or pct_counts_ribo contains NA/non-numeric values after conversion.")

mt_cutoff = obs[CONDITION_COL].map(cond_pct_mt_cutofs).astype(float)
ribo_cutoff = obs[CONDITION_COL].map(cond_pct_ribo_cutofs).astype(float)

qc_fail = (mt >= mt_cutoff) | (ribo <= ribo_cutoff)
keep = ~qc_fail
keep_mask = keep.to_numpy()

print(f"Cells before filtering: {adata.n_obs:,}")
print(f"Cells removed:          {int(qc_fail.sum()):,} ({qc_fail.mean():.2%})")
print(f"Cells kept:             {int(keep.sum()):,} ({keep.mean():.2%})")

filter_annot = obs[[CONDITION_COL, SAMPLE_COL, "pct_counts_mt", "pct_counts_ribo"]].copy()
filter_annot["mt_cutoff"] = mt_cutoff.to_numpy()
filter_annot["ribo_cutoff"] = ribo_cutoff.to_numpy()
filter_annot["qc_fail"] = qc_fail.to_numpy()
filter_annot["qc_keep"] = keep.to_numpy()

filter_summary_condition = (
    filter_annot
    .groupby(CONDITION_COL, observed=True)
    .agg(
        n_cells=("qc_fail", "size"),
        n_removed=("qc_fail", "sum"),
        frac_removed=("qc_fail", "mean"),
        mt_cutoff=("mt_cutoff", "first"),
        ribo_cutoff=("ribo_cutoff", "first"),
        median_pct_mt=("pct_counts_mt", "median"),
        median_pct_ribo=("pct_counts_ribo", "median"),
    )
)
filter_summary_condition["n_kept"] = filter_summary_condition["n_cells"] - filter_summary_condition["n_removed"]
filter_summary_condition = filter_summary_condition.loc[[c for c in condition_order if c in filter_summary_condition.index]]
show_df(filter_summary_condition, n=100)

filter_summary_sample = (
    filter_annot
    .groupby([CONDITION_COL, SAMPLE_COL], observed=True)
    .agg(
        n_cells=("qc_fail", "size"),
        n_removed=("qc_fail", "sum"),
        frac_removed=("qc_fail", "mean"),
    )
)
filter_summary_sample["n_kept"] = filter_summary_sample["n_cells"] - filter_summary_sample["n_removed"]
show_df(filter_summary_sample.reset_index(), n=100)

filter_summary_condition.to_csv(PLOT_OUT_DIR / "qc_filter_summary_by_condition.tsv", sep="\t")
filter_summary_sample.to_csv(PLOT_OUT_DIR / "qc_filter_summary_by_condition_and_sample_name.tsv", sep="\t")

# Backed AnnData/MuData filtered views for downstream code, without loading adata.X.
adata_f = adata[keep_mask, :]
print("Filtered AnnData view:", adata_f)

mdata_f = None
if mdata is not None:
    try:
        if len(mdata.obs_names) == len(keep_mask) and np.array_equal(
            pd.Index(mdata.obs_names).astype(str).to_numpy(),
            pd.Index(adata.obs_names).astype(str).to_numpy(),
        ):
            mdata_f = mdata[keep_mask, :]
            print("Filtered MuData view:", mdata_f)
        else:
            print("MuData obs_names do not exactly match AnnData obs_names; not creating mdata_f view.")
    except Exception as e:
        print("Could not create mdata_f view; heatmap does not require it.")
        print(type(e).__name__, e)

# %% [markdown]
# ## Load projected cNMF scores and zero-based program names

# %%
if SCORE_KEY not in adata.obsm:
    raise KeyError(f"{SCORE_KEY!r} is not in adata.obsm. Available obsm keys: {list(adata.obsm.keys())}")

usage = np.asarray(adata.obsm[SCORE_KEY])
if usage.ndim != 2:
    raise ValueError(f"Expected {SCORE_KEY} to be 2D, got shape {usage.shape}")

n_cells, n_programs = usage.shape
print("usage shape:", usage.shape)
assert n_cells == adata.n_obs

# Program names from uns, with robust zero-based fallback.
projection_uns = adata.uns.get("perturbseq_cNMF_projection", {}) if hasattr(adata, "uns") else {}
program_names = None
if isinstance(projection_uns, dict) and "program_names" in projection_uns:
    program_names = [str(x) for x in list(projection_uns["program_names"])]

if program_names is None or len(program_names) != n_programs:
    score_cols = [c for c in obs.columns if c.startswith("perturbseq_cNMF_P") and c.endswith("_usage")]
    if len(score_cols) == n_programs:
        def score_col_to_program(c):
            return c.replace("perturbseq_cNMF_", "").replace("_usage", "")
        score_cols = sorted(score_cols, key=lambda c: int(score_col_to_program(c).replace("P", "")))
        program_names = [score_col_to_program(c) for c in score_cols]
    else:
        program_names = [f"P{i}" for i in range(n_programs)]

expected_zero = [f"P{i}" for i in range(n_programs)]
expected_one = [f"P{i}" for i in range(1, n_programs + 1)]
if program_names == expected_one:
    print("WARNING: program names look one-based. Relabeling display names to zero-based P0...P49.")
    program_names = expected_zero
elif program_names != expected_zero:
    print("WARNING: program names are not exactly P0...P49. Using names from file:")
    print(program_names)

print("Program names:", program_names[:10], "...", program_names[-5:])

usage_f = usage[keep_mask, :]
obs_f = obs.loc[keep_mask].copy()
print("Filtered usage shape:", usage_f.shape)

# Basic sanity checks.
print("NaN count:", np.isnan(usage_f).sum())
print("Inf count:", np.isinf(usage_f).sum())
print("negative count:", (usage_f < -1e-8).sum())
print("row sum summary:")
print(pd.Series(usage_f.sum(axis=1)).describe())

# %% [markdown]
# ## Attach program labels from program metadata

# %%
label_map = {}
if PROGRAM_META_PATH.exists():
    prog_meta = pd.read_csv(PROGRAM_META_PATH)
    if {"program_name", "program_label"}.issubset(prog_meta.columns):
        tmp = prog_meta[["program_name", "program_label"]].drop_duplicates("program_name").copy()
        tmp["program_name"] = tmp["program_name"].astype(int)
        label_map = {f"P{int(r.program_name)}": str(r.program_label) for r in tmp.itertuples(index=False)}
        print(f"Loaded program labels from {PROGRAM_META_PATH}")
    else:
        print("Program metadata missing program_name/program_label columns; using P labels only.")
else:
    print("Program metadata file not found; using P labels only:", PROGRAM_META_PATH)


def shorten_label(x, max_chars=MAX_PROGRAM_LABEL_CHARS):
    x = str(x)
    if len(x) <= max_chars:
        return x
    return x[: max_chars - 1] + "…"

program_display_names = []
for p in program_names:
    label = label_map.get(p, "")
    if SHOW_PROGRAM_LABELS and label and label.lower() != "nan":
        program_display_names.append(f"{p} | {shorten_label(label)}")
    else:
        program_display_names.append(p)

program_label_df = pd.DataFrame({
    "program": program_names,
    "display_name": program_display_names,
    "program_label": [label_map.get(p, "") for p in program_names],
})
show_df(program_label_df, n=100)
program_label_df.to_csv(PLOT_OUT_DIR / "zero_based_program_labels_used_in_heatmap.tsv", sep="\t", index=False)

# %% [markdown]
# ## Compute mean projected cNMF usage by sample_name

# %%
if SAMPLE_COL not in obs_f.columns:
    raise KeyError(f"{SAMPLE_COL!r} not found in filtered obs.")

score_df = pd.DataFrame(usage_f, index=obs_f.index, columns=program_names)
score_df[SAMPLE_COL] = obs_f[SAMPLE_COL].astype(str).to_numpy()
score_df[CONDITION_COL] = obs_f[CONDITION_COL].astype(str).to_numpy()

group_mean_sample_x_program = score_df.groupby(SAMPLE_COL, observed=True)[program_names].mean()
group_median_sample_x_program = score_df.groupby(SAMPLE_COL, observed=True)[program_names].median()
group_n = score_df.groupby(SAMPLE_COL, observed=True).size().rename("n_cells")

# Order samples by condition order, then sample_name.
def mode_or_first(values):
    vc = pd.Series(values).value_counts(dropna=False)
    return str(vc.index[0]) if len(vc) else ""

sample_condition = score_df.groupby(SAMPLE_COL, observed=True)[CONDITION_COL].agg(mode_or_first)
sample_order_df = pd.DataFrame({
    SAMPLE_COL: group_mean_sample_x_program.index.astype(str),
})
sample_order_df[CONDITION_COL] = sample_order_df[SAMPLE_COL].map(sample_condition)
sample_order_df["condition_rank"] = sample_order_df[CONDITION_COL].map({c: i for i, c in enumerate(condition_order)})
sample_order_df["condition_rank"] = sample_order_df["condition_rank"].fillna(len(condition_order)).astype(int)
sample_order_df = sample_order_df.sort_values(["condition_rank", SAMPLE_COL])
sample_order = sample_order_df[SAMPLE_COL].tolist()

heatmap_mean = group_mean_sample_x_program.loc[sample_order, program_names].T
heatmap_median = group_median_sample_x_program.loc[sample_order, program_names].T
sample_n_ordered = group_n.loc[sample_order]

print("Heatmap mean matrix, programs x sample_name:", heatmap_mean.shape)
show_df(sample_order_df, n=100)
show_df(heatmap_mean.iloc[:10, :10], n=20)

heatmap_mean.to_csv(PLOT_OUT_DIR / "projected_cNMF_mean_usage_by_sample_name.filtered.tsv", sep="\t")
heatmap_median.to_csv(PLOT_OUT_DIR / "projected_cNMF_median_usage_by_sample_name.filtered.tsv", sep="\t")
sample_n_ordered.to_csv(PLOT_OUT_DIR / "filtered_cells_per_sample_name.tsv", sep="\t", header=True)

# %% [markdown]
# ## Make heatmap

# %%
if HEATMAP_SCALE == "zscore_by_program":
    row_mean = heatmap_mean.mean(axis=1)
    row_std = heatmap_mean.std(axis=1).replace(0, np.nan)
    heatmap_plot = heatmap_mean.sub(row_mean, axis=0).div(row_std, axis=0).fillna(0)
    colorbar_label = "Mean projected cNMF usage, z-scored within program"
    file_suffix = "mean_usage_zscore_by_program"
elif HEATMAP_SCALE == "mean":
    heatmap_plot = heatmap_mean.copy()
    colorbar_label = "Mean projected cNMF usage"
    file_suffix = "mean_usage"
else:
    raise ValueError(f"Unknown HEATMAP_SCALE: {HEATMAP_SCALE}")

heatmap_plot.to_csv(PLOT_OUT_DIR / f"projected_cNMF_by_sample_name.filtered.{file_suffix}.tsv", sep="\t")

n_programs, n_samples = heatmap_plot.shape
fig_w = max(8, 0.42 * n_samples + 2.5)
fig_h = max(10, 0.24 * n_programs + 1.5)

xlabels = [f"{s}\n(n={int(sample_n_ordered.loc[s]):,})" for s in heatmap_plot.columns]

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
im = ax.imshow(heatmap_plot.to_numpy(), aspect="auto", interpolation="nearest")

ax.set_xticks(np.arange(n_samples))
ax.set_xticklabels(xlabels, rotation=90, ha="center", va="top", fontsize=6)
ax.set_yticks(np.arange(n_programs))
ax.set_yticklabels(program_display_names, fontsize=5)

ax.set_xlabel(SAMPLE_COL)
ax.set_ylabel("Projected perturb-seq cNMF program")
ax.set_title(f"Projected perturb-seq cNMF programs by {SAMPLE_COL}\nQC-filtered cells only")

cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.015)
cbar.set_label(colorbar_label)

fig.tight_layout()

base = PLOT_OUT_DIR / f"heatmap_projected_cNMF_by_{SAMPLE_COL}.filtered.{file_suffix}.all_program_names"
fig.savefig(base.with_suffix(".png"), bbox_inches="tight")
fig.savefig(base.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(base.with_suffix(".svg"), bbox_inches="tight")
print("Saved:")
print(base.with_suffix(".png"))
print(base.with_suffix(".pdf"))
print(base.with_suffix(".svg"))
plt.show()

# %% [markdown]
# ## Optional: show the unscaled mean usage heatmap too

# %%
if HEATMAP_SCALE != "mean":
    heatmap_plot_unscaled = heatmap_mean.copy()
    heatmap_plot_unscaled.to_csv(PLOT_OUT_DIR / "projected_cNMF_by_sample_name.filtered.mean_usage.tsv", sep="\t")

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    im = ax.imshow(heatmap_plot_unscaled.to_numpy(), aspect="auto", interpolation="nearest")

    ax.set_xticks(np.arange(n_samples))
    ax.set_xticklabels(xlabels, rotation=90, ha="center", va="top", fontsize=6)
    ax.set_yticks(np.arange(n_programs))
    ax.set_yticklabels(program_display_names, fontsize=5)

    ax.set_xlabel(SAMPLE_COL)
    ax.set_ylabel("Projected perturb-seq cNMF program")
    ax.set_title(f"Projected perturb-seq cNMF programs by {SAMPLE_COL}\nQC-filtered cells only; unscaled mean usage")

    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.015)
    cbar.set_label("Mean projected cNMF usage")

    fig.tight_layout()

    base_unscaled = PLOT_OUT_DIR / f"heatmap_projected_cNMF_by_{SAMPLE_COL}.filtered.mean_usage.all_program_names"
    fig.savefig(base_unscaled.with_suffix(".png"), bbox_inches="tight")
    fig.savefig(base_unscaled.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(base_unscaled.with_suffix(".svg"), bbox_inches="tight")
    print("Saved unscaled heatmap:")
    print(base_unscaled.with_suffix(".png"))
    print(base_unscaled.with_suffix(".pdf"))
    print(base_unscaled.with_suffix(".svg"))
    plt.show()

# %% [markdown]
# ## Close backed files

# %%
try:
    adata.file.close()
except Exception:
    pass

if mdata is not None:
    try:
        mdata.file.close()
    except Exception:
        pass

print("Done.")


In [ ]:
# %%
from pathlib import Path
import re
import textwrap
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt

# Keep text editable in Illustrator SVG output
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# %%
OUT_DIR = Path("/scratch/users/edtng/20260325_Hippo_timecourse/perturbseq_cNMF_projection")
H5AD_PATH = OUT_DIR / "Hippo_timecourse_combined.projected_perturbseq_cNMF_50_dt_0_2.zero_based.h5ad"
PROGRAM_META_PATH = Path("/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/data/program_activity_long_v4.csv")
HEATMAP_OUT_DIR = OUT_DIR / "heatmaps_after_qc_filter_redblue_svg_compact_ordered"
HEATMAP_OUT_DIR.mkdir(parents=True, exist_ok=True)

SCORE_KEY = "X_perturbseq_cNMF_usage"  # normalized projected usage
SAMPLE_COL = "sample_name"
CONDITION_COL = "condition"

# More compact label options
PROGRAM_LABEL_MAX_CHARS = 32
PROGRAM_LABEL_WRAP = 24  # set None to disable wrapping
PROGRAM_LABEL_MODE = "compact"  # compact / full / id_only

# Wider canvas options for long sample names
BASE_FIG_W = 16
PER_SAMPLE_W = 0.9
BASE_FIG_H = 14
PER_PROGRAM_H = 0.28
X_LABEL_ROTATION = 50
X_LABEL_HA = "right"

# Red-blue diverging heatmap palette
ZSCORE_CMAP = "RdBu_r"
RAW_CMAP = "RdBu_r"

# Exact time-course order at the condition level
CONDITION_TIMECOURSE_ORDER = [
    'D0_hPSC',
    'D1_MPS',
    'D1_MPS+TDI',
    'D1_MPS+IAG',
    'D1.33_LM',
    'D1.33_LM+TDI',
    'D1.33_LM+IAG',
    'D1.67_LM',
    'D1.67_LM+TDI',
    'D1.67_LM+IAG',
    'D2_LM',
    'D2_LM+TDI',
    'D2_LM+IAG',
    'D2_LM+TDI->D3_AE+TDI',
    'D2_LM+IAG->D3_AE+IAG',
    'D3_AE',
    'D3_AE+TDI',
    'D3_AE+IAG',
]

# %%
cond_pct_ribo_cutofs = {
    'D0_hPSC': 2.4,
    'D1_MPS': 2.3,
    'D1_MPS+TDI': 2.5,
    'D1_MPS+IAG': 2.2,
    'D1.33_LM': 1.9,
    'D1.33_LM+TDI': 2.1,
    'D1.33_LM+IAG': 2.2,
    'D1.67_LM': 2.0,
    'D1.67_LM+TDI': 2.2,
    'D1.67_LM+IAG': 1.9,
    'D2_LM': 2.0,
    'D2_LM+TDI': 2.4,
    'D2_LM+IAG': 1.9,
    'D3_AE': 1.0,
    'D3_AE+TDI': 2.0,
    'D3_AE+IAG': 1.0,
    'D2_LM+TDI->D3_AE+TDI': 2.5,
    'D2_LM+IAG->D3_AE+IAG': 1.0
}

cond_pct_mt_cutofs = {
    'D0_hPSC': 5.5,
    'D1_MPS': 3.0,
    'D1_MPS+TDI': 3.0,
    'D1_MPS+IAG': 4,
    'D1.33_LM': 3.5,
    'D1.33_LM+TDI': 3.75,
    'D1.33_LM+IAG': 5,
    'D1.67_LM': 4.0,
    'D1.67_LM+TDI': 5.5,
    'D1.67_LM+IAG': 3.5,
    'D2_LM': 6.5,
    'D2_LM+TDI': 8,
    'D2_LM+IAG': 3.5,
    'D3_AE': 3.25,
    'D3_AE+TDI': 7.0,
    'D3_AE+IAG': 3.75,
    'D2_LM+TDI->D3_AE+TDI': 7,
    'D2_LM+IAG->D3_AE+IAG': 3.75
}

# %%
def infer_condition_from_sample_name(sample_name: str):
    sample_name = str(sample_name)
    # Exact match first
    if sample_name in cond_pct_mt_cutofs:
        return sample_name
    # Otherwise find the longest condition key contained in sample_name
    hits = [cond for cond in cond_pct_mt_cutofs if cond in sample_name]
    if not hits:
        return None
    return sorted(hits, key=len, reverse=True)[0]


def compact_program_label(program_id: str, raw_label: str):
    if PROGRAM_LABEL_MODE == "id_only":
        label = str(program_id)
    elif PROGRAM_LABEL_MODE == "full":
        label = f"{program_id}: {raw_label}"
    else:
        # compact
        label = f"{program_id} {raw_label}"
    label = re.sub(r"\s+", " ", str(label)).strip()
    if PROGRAM_LABEL_MAX_CHARS is not None and len(label) > PROGRAM_LABEL_MAX_CHARS:
        label = label[: PROGRAM_LABEL_MAX_CHARS - 1].rstrip() + "…"
    if PROGRAM_LABEL_WRAP:
        label = "\n".join(textwrap.wrap(label, width=PROGRAM_LABEL_WRAP, break_long_words=False))
    return label


# %%
adata = sc.read_h5ad(H5AD_PATH)
print(adata)
print("obs columns:", list(adata.obs.columns))
print("obsm keys:", list(adata.obsm.keys()))

# %%
# Derive or use condition column
if CONDITION_COL not in adata.obs.columns:
    if SAMPLE_COL not in adata.obs.columns:
        raise ValueError(f"Neither `{CONDITION_COL}` nor `{SAMPLE_COL}` found in adata.obs")
    inferred = adata.obs[SAMPLE_COL].astype(str).map(infer_condition_from_sample_name)
    if inferred.isna().any():
        bad = adata.obs.loc[inferred.isna(), SAMPLE_COL].astype(str).unique().tolist()
        raise ValueError(
            f"Could not infer condition from some sample names. Example unmatched sample_name values: {bad[:10]}"
        )
    adata.obs[CONDITION_COL] = inferred.to_numpy()
    print(f"`{CONDITION_COL}` not found; inferred from `{SAMPLE_COL}`.")

# Validate condition coverage for filtering dictionaries
conditions_present = pd.Index(adata.obs[CONDITION_COL].astype(str).unique())
missing_mt = sorted(set(conditions_present) - set(cond_pct_mt_cutofs))
missing_ribo = sorted(set(conditions_present) - set(cond_pct_ribo_cutofs))
if missing_mt or missing_ribo:
    raise ValueError(
        "Some conditions are missing from the QC cutoff dictionaries.\n"
        f"Missing in cond_pct_mt_cutofs: {missing_mt}\n"
        f"Missing in cond_pct_ribo_cutofs: {missing_ribo}"
    )

# %%
# Apply the requested condition-specific QC filters
keep_mask = np.ones(adata.n_obs, dtype=bool)
summary_records = []
summary_sample_records = []
cond_series = adata.obs[CONDITION_COL].astype(str)
sample_series = adata.obs[SAMPLE_COL].astype(str)

for cond in cond_series.value_counts().index:
    cond_mask = cond_series.to_numpy() == cond
    fail_mask = cond_mask & (
        (adata.obs["pct_counts_mt"].to_numpy() >= cond_pct_mt_cutofs[cond]) |
        (adata.obs["pct_counts_ribo"].to_numpy() <= cond_pct_ribo_cutofs[cond])
    )
    keep_mask[fail_mask] = False

    n_total = int(cond_mask.sum())
    n_remove = int(fail_mask.sum())
    n_keep = n_total - n_remove
    summary_records.append({
        "condition": cond,
        "n_total": n_total,
        "n_remove": n_remove,
        "n_keep": n_keep,
        "frac_remove": n_remove / n_total if n_total > 0 else np.nan,
        "pct_counts_mt_cutoff": cond_pct_mt_cutofs[cond],
        "pct_counts_ribo_cutoff": cond_pct_ribo_cutofs[cond],
    })

    for sample_name in sample_series[cond_mask].value_counts().index:
        sub_mask = cond_mask & (sample_series.to_numpy() == sample_name)
        sub_fail = fail_mask & (sample_series.to_numpy() == sample_name)
        n_sub_total = int(sub_mask.sum())
        n_sub_remove = int(sub_fail.sum())
        summary_sample_records.append({
            "condition": cond,
            SAMPLE_COL: sample_name,
            "n_total": n_sub_total,
            "n_remove": n_sub_remove,
            "n_keep": n_sub_total - n_sub_remove,
            "frac_remove": n_sub_remove / n_sub_total if n_sub_total > 0 else np.nan,
        })

adata_f = adata[keep_mask].copy()
print(f"Cells before filter: {adata.n_obs:,}")
print(f"Cells after filter:  {adata_f.n_obs:,}")

qc_summary_df = pd.DataFrame(summary_records)
qc_summary_sample_df = pd.DataFrame(summary_sample_records)
qc_summary_df.to_csv(HEATMAP_OUT_DIR / "qc_filter_summary_by_condition.tsv", sep="\t", index=False)
qc_summary_sample_df.to_csv(HEATMAP_OUT_DIR / f"qc_filter_summary_by_condition_and_{SAMPLE_COL}.tsv", sep="\t", index=False)

# %%
# Pull projected usage matrix after filtering
if SCORE_KEY not in adata_f.obsm:
    raise KeyError(f"{SCORE_KEY} not found in adata.obsm")

usage = np.asarray(adata_f.obsm[SCORE_KEY])
print("usage shape:", usage.shape)

# Program names from saved projection metadata if available; otherwise infer P0..P49
program_names = None
if "perturbseq_cNMF_projection" in adata_f.uns:
    meta_uns = adata_f.uns["perturbseq_cNMF_projection"]
    if isinstance(meta_uns, dict) and "program_names" in meta_uns:
        program_names = [str(x) for x in meta_uns["program_names"]]

if program_names is None:
    program_names = [f"P{i}" for i in range(usage.shape[1])]

if len(program_names) != usage.shape[1]:
    raise ValueError(
        f"Number of program names ({len(program_names)}) does not match usage columns ({usage.shape[1]})."
    )

# %%
# Load human-readable program labels if available
program_display_map = {p: p for p in program_names}
program_label_full_map = {p: p for p in program_names}
if PROGRAM_META_PATH.exists():
    prog_meta = pd.read_csv(PROGRAM_META_PATH).drop_duplicates("program_name")
    if "program_name" in prog_meta.columns:
        for _, row in prog_meta.iterrows():
            pid = int(row["program_name"])
            pname = f"P{pid}"
            if pname in program_display_map:
                if "program_label" in prog_meta.columns and pd.notna(row["program_label"]):
                    raw_label = str(row["program_label"])
                else:
                    raw_label = pname
                program_label_full_map[pname] = raw_label
                program_display_map[pname] = compact_program_label(pname, raw_label)

label_table = pd.DataFrame({
    "program": program_names,
    "full_program_label": [program_label_full_map[p] for p in program_names],
    "display_label": [program_display_map[p] for p in program_names],
})
label_table.to_csv(HEATMAP_OUT_DIR / "zero_based_program_labels_used_in_heatmap.tsv", sep="\t", index=False)

# %%
# Make per-cell score dataframe and aggregate by sample_name
if SAMPLE_COL not in adata_f.obs.columns:
    raise KeyError(f"{SAMPLE_COL} not found in adata.obs")

score_df = pd.DataFrame(usage, index=adata_f.obs_names, columns=program_names)
score_df[SAMPLE_COL] = adata_f.obs[SAMPLE_COL].astype(str).to_numpy()
score_df[CONDITION_COL] = adata_f.obs[CONDITION_COL].astype(str).to_numpy()

# Order sample columns exactly by time course using condition order, then preserve within-condition sample order as observed
observed_pairs = score_df[[CONDITION_COL, SAMPLE_COL]].drop_duplicates()
observed_sample_order = []
for cond in CONDITION_TIMECOURSE_ORDER:
    sub = observed_pairs.loc[observed_pairs[CONDITION_COL] == cond, SAMPLE_COL].tolist()
    observed_sample_order.extend(sub)
# Append any unexpected conditions/samples afterward in first-observed order
remaining = [s for s in observed_pairs[SAMPLE_COL].tolist() if s not in observed_sample_order]
sample_order = observed_sample_order + remaining

sample_order_df = pd.DataFrame({
    SAMPLE_COL: sample_order,
    CONDITION_COL: [
        score_df.loc[score_df[SAMPLE_COL] == s, CONDITION_COL].iloc[0] for s in sample_order
    ],
    "sample_order_index": np.arange(len(sample_order)),
})
sample_order_df.to_csv(HEATMAP_OUT_DIR / f"{SAMPLE_COL}_timecourse_order_used.tsv", sep="\t", index=False)

# Aggregate
mean_by_sample = score_df.groupby(SAMPLE_COL)[program_names].mean().reindex(sample_order)
median_by_sample = score_df.groupby(SAMPLE_COL)[program_names].median().reindex(sample_order)
n_by_sample = score_df.groupby(SAMPLE_COL).size().rename("n_cells").reindex(sample_order)

mean_by_sample.to_csv(HEATMAP_OUT_DIR / f"projected_cNMF_mean_usage_by_{SAMPLE_COL}.filtered.tsv", sep="\t")
median_by_sample.to_csv(HEATMAP_OUT_DIR / f"projected_cNMF_median_usage_by_{SAMPLE_COL}.filtered.tsv", sep="\t")
n_by_sample.to_csv(HEATMAP_OUT_DIR / f"projected_cNMF_ncells_by_{SAMPLE_COL}.filtered.tsv", sep="\t")

print(mean_by_sample.shape)
display(sample_order_df)
display(n_by_sample.to_frame())
display(mean_by_sample.head())

# %%
# Build heatmap matrices
heatmap_raw = mean_by_sample.T.copy()  # programs x sample_name
heatmap_raw.index = [program_display_map[p] for p in heatmap_raw.index]

heatmap_z = heatmap_raw.sub(heatmap_raw.mean(axis=1), axis=0)
heatmap_z = heatmap_z.div(heatmap_raw.std(axis=1).replace(0, np.nan), axis=0)

heatmap_raw.to_csv(HEATMAP_OUT_DIR / f"projected_cNMF_mean_usage_by_{SAMPLE_COL}.filtered.programs_x_samples.tsv", sep="\t")
heatmap_z.to_csv(HEATMAP_OUT_DIR / f"projected_cNMF_mean_usage_zscore_by_program_by_{SAMPLE_COL}.filtered.programs_x_samples.tsv", sep="\t")

# %%
def plot_heatmap_pcolormesh(df, title, out_prefix, cmap="RdBu_r", center_zero=False):
    mat = df.to_numpy(dtype=float)
    n_rows, n_cols = mat.shape

    # Symmetric color range for z-scored heatmaps; regular min-max for raw
    if center_zero:
        vmax = np.nanmax(np.abs(mat))
        vmin = -vmax
    else:
        vmin = np.nanmin(mat)
        vmax = np.nanmax(mat)
        if np.isclose(vmin, vmax):
            vmax = vmin + 1e-9

    fig_w = max(BASE_FIG_W, PER_SAMPLE_W * n_cols + 8)
    fig_h = max(BASE_FIG_H, PER_PROGRAM_H * n_rows + 4)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    mesh = ax.pcolormesh(
        np.arange(n_cols + 1),
        np.arange(n_rows + 1),
        mat,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        shading="flat",
        linewidth=0,
        antialiased=False,
    )

    ax.set_xlim(0, n_cols)
    ax.set_ylim(n_rows, 0)
    ax.set_xticks(np.arange(n_cols) + 0.5)
    ax.set_yticks(np.arange(n_rows) + 0.5)
    ax.set_xticklabels(df.columns.tolist(), rotation=X_LABEL_ROTATION, ha=X_LABEL_HA)
    ax.set_yticklabels(df.index.tolist(), fontsize=9)
    ax.set_xlabel(SAMPLE_COL)
    ax.set_ylabel("Projected perturb-seq cNMF program")
    ax.set_title(title)

    cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
    cbar.set_label("z-score across samples" if center_zero else "mean projected usage")

    fig.subplots_adjust(left=0.28, bottom=0.22, right=0.96, top=0.96)

    for ext in ["png", "pdf", "svg"]:
        fig.savefig(
            HEATMAP_OUT_DIR / f"{out_prefix}.{ext}",
            dpi=300 if ext == "png" else None,
            bbox_inches="tight",
        )

    plt.show()
    plt.close(fig)

# %%
# Main z-scored red-blue heatmap
plot_heatmap_pcolormesh(
    heatmap_z,
    title=f"Projected perturb-seq cNMF by {SAMPLE_COL} (filtered; z-scored by program)",
    out_prefix=f"heatmap_projected_cNMF_by_{SAMPLE_COL}.filtered.mean_usage_zscore_by_program.all_program_names.redblue_editable.compact_ordered",
    cmap=ZSCORE_CMAP,
    center_zero=True,
)

# %%
# Raw mean-usage red-blue heatmap
plot_heatmap_pcolormesh(
    heatmap_raw,
    title=f"Projected perturb-seq cNMF by {SAMPLE_COL} (filtered; raw mean usage)",
    out_prefix=f"heatmap_projected_cNMF_by_{SAMPLE_COL}.filtered.mean_usage.all_program_names.redblue_editable.compact_ordered",
    cmap=RAW_CMAP,
    center_zero=False,
)

# %%
print("Saved outputs to:")
print(HEATMAP_OUT_DIR)
for p in sorted(HEATMAP_OUT_DIR.glob("*")):
    print(p.name)


#### DE analysis

In [ ]:
# %% [markdown]
# # Hippo time-course matched pseudobulk DE + projected cNMF program contrasts
#
# Copy these cells into a Jupyter notebook and run in order.
#
# This workflow:
# 1. Loads the zero-based projected AnnData.
# 2. Applies the condition-specific mito/ribo filters.
# 3. Creates sample-level pseudobulk raw counts.
# 4. Runs PyDESeq2 for matched treated-vs-control gene DE.
# 5. Runs matched sample-level tests on projected cNMF program scores.
# 6. Optionally scores cNMF program enrichment in DE gene log2FC.

# %%
# CELL 1 — Imports, paths, sample metadata, QC cutoffs, and contrasts

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse, stats
import matplotlib as mpl
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# Input/output paths
H5AD_PATH = Path(
    "/scratch/users/edtng/20260325_Hippo_timecourse/perturbseq_cNMF_projection/"
    "Hippo_timecourse_combined.projected_perturbseq_cNMF_50_dt_0_2.zero_based.h5ad"
)

BASE_OUT = Path("/scratch/users/edtng/20260325_Hippo_timecourse/perturbseq_cNMF_matched_DE")
BASE_OUT.mkdir(parents=True, exist_ok=True)

SPECTRA_TPM_PATH = Path(
    "/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result/"
    "030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50/"
    "030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50."
    "gene_spectra_tpm.k_50.dt_0_2.txt"
)

# Use raw counts for DE. Change to layer:mature or layer:nascent if you want those layers.
COUNTS_SOURCE = "X"  # "X" or "layer:<layer_name>"
PROGRAM_SCORE_KEY = "X_perturbseq_cNMF_usage"  # normalized projected usage
SAMPLE_COL = "sample_name"
CHUNK_SIZE = 4000
GENE_SYMBOL_COL = "gene"

# PyDESeq2 settings
N_CPUS = 8
MIN_SAMPLES_PER_GROUP = 2
MIN_COUNT = 10
MIN_SAMPLES_EXPRESSED = 2
SIZE_FACTORS_FIT_TYPE = "ratio"  # ratio / poscounts / iterative
USE_PAIRED_REPLICATE_DESIGN = True
QUIET_PYDESEQ2 = False

# Programs to highlight in the program-score contrast output. Zero-based labels.
KEY_PROGRAMS = ["P15", "P17", "P34"]

sample_mapping_dict = {
    '1-1': 'D0_hPSC_rep1', '1-2': 'D0_hPSC_rep2', '1-3': 'D0_hPSC_rep3',
    '1-4': 'D1_MPS_rep1', '1-5': 'D1_MPS_rep2', '1-6': 'D1_MPS_rep3',
    '1-7': 'D1_MPS+TDI_rep1', '1-8': 'D1_MPS+TDI_rep2', '1-9': 'D1_MPS+TDI_rep3',
    '1-10': 'D1_MPS+IAG_rep1', '1-11': 'D1_MPS+IAG_rep2', '1-12': 'D1_MPS+IAG_rep3',
    '2-1': 'D1.33_LM_rep1', '2-2': 'D1.33_LM_rep2', '2-3': 'D1.33_LM_rep3',
    '2-4': 'D1.33_LM+TDI_rep1', '2-5': 'D1.33_LM+TDI_rep2', '2-6': 'D1.33_LM+TDI_rep3',
    '2-7': 'D1.33_LM+IAG_rep1', '2-8': 'D1.33_LM+IAG_rep2', '2-9': 'D1.33_LM+IAG_rep3',
    '3-1': 'D1.67_LM_rep1', '3-2': 'D1.67_LM_rep2', '3-3': 'D1.67_LM_rep3',
    '3-4': 'D1.67_LM+TDI_rep1', '3-5': 'D1.67_LM+TDI_rep2', '3-6': 'D1.67_LM+TDI_rep3',
    '3-7': 'D1.67_LM+IAG_rep1', '3-8': 'D1.67_LM+IAG_rep2', '3-9': 'D1.67_LM+IAG_rep3',
    '4-1': 'D2_LM_rep1', '4-2': 'D2_LM_rep2', '4-3': 'D2_LM_rep3',
    '4-4': 'D2_LM+TDI_rep1', '4-5': 'D2_LM+TDI_rep2', '4-6': 'D2_LM+TDI_rep3',
    '4-7': 'D2_LM+IAG_rep1', '4-8': 'D2_LM+IAG_rep2', '4-9': 'D2_LM+IAG_rep3',
    '5-1': 'D3_AE_rep1', '5-2': 'D3_AE_rep2', '5-3': 'D3_AE_rep3',
    '5-4': 'D3_AE+TDI_rep1', '5-5': 'D3_AE+TDI_rep2', '5-6': 'D3_AE+TDI_rep3',
    '5-7': 'D3_AE+IAG_rep1', '5-8': 'D3_AE+IAG_rep2', '5-9': 'D3_AE+IAG_rep3',
    '5-10': 'D2_LM+TDI->D3_AE+TDI_rep1', '5-11': 'D2_LM+TDI->D3_AE+TDI_rep2',
    '5-12': 'D2_LM+TDI->D3_AE+TDI_rep3', '5-13': 'D2_LM+IAG->D3_AE+IAG_rep1',
    '5-14': 'D2_LM+IAG->D3_AE+IAG_rep2', '5-15': 'D2_LM+IAG->D3_AE+IAG_rep3',
}

cond_pct_ribo_cutoffs = {
    'D0_hPSC': 2.4,
    'D1_MPS': 2.3,
    'D1_MPS+TDI': 2.5,
    'D1_MPS+IAG': 2.2,
    'D1.33_LM': 1.9,
    'D1.33_LM+TDI': 2.1,
    'D1.33_LM+IAG': 2.2,
    'D1.67_LM': 2.0,
    'D1.67_LM+TDI': 2.2,
    'D1.67_LM+IAG': 1.9,
    'D2_LM': 2.0,
    'D2_LM+TDI': 2.4,
    'D2_LM+IAG': 1.9,
    'D3_AE': 1.0,
    'D3_AE+TDI': 2.0,
    'D3_AE+IAG': 1.0,
    'D2_LM+TDI->D3_AE+TDI': 2.5,
    'D2_LM+IAG->D3_AE+IAG': 1.0,
}

cond_pct_mt_cutoffs = {
    'D0_hPSC': 5.5,
    'D1_MPS': 3.0,
    'D1_MPS+TDI': 3.0,
    'D1_MPS+IAG': 4,
    'D1.33_LM': 3.5,
    'D1.33_LM+TDI': 3.75,
    'D1.33_LM+IAG': 5,
    'D1.67_LM': 4.0,
    'D1.67_LM+TDI': 5.5,
    'D1.67_LM+IAG': 3.5,
    'D2_LM': 6.5,
    'D2_LM+TDI': 8,
    'D2_LM+IAG': 3.5,
    'D3_AE': 3.25,
    'D3_AE+TDI': 7.0,
    'D3_AE+IAG': 3.75,
    'D2_LM+TDI->D3_AE+TDI': 7,
    'D2_LM+IAG->D3_AE+IAG': 3.75,
}

# Same-timepoint treated vs control contrasts.
contrasts = pd.DataFrame([
    ('D1_MPS+TDI', 'D1_MPS', 'TDI_at_D1_MPS'),
    ('D1_MPS+IAG', 'D1_MPS', 'IAG_at_D1_MPS'),
    ('D1.33_LM+TDI', 'D1.33_LM', 'TDI_at_D1.33_LM'),
    ('D1.33_LM+IAG', 'D1.33_LM', 'IAG_at_D1.33_LM'),
    ('D1.67_LM+TDI', 'D1.67_LM', 'TDI_at_D1.67_LM'),
    ('D1.67_LM+IAG', 'D1.67_LM', 'IAG_at_D1.67_LM'),
    ('D2_LM+TDI', 'D2_LM', 'TDI_at_D2_LM'),
    ('D2_LM+IAG', 'D2_LM', 'IAG_at_D2_LM'),
    ('D3_AE+TDI', 'D3_AE', 'TDI_at_D3_AE'),
    ('D3_AE+IAG', 'D3_AE', 'IAG_at_D3_AE'),
    ('D2_LM+TDI->D3_AE+TDI', 'D3_AE', 'TDI_D2_to_D3_vs_D3_AE'),
    ('D2_LM+IAG->D3_AE+IAG', 'D3_AE', 'IAG_D2_to_D3_vs_D3_AE'),
], columns=['treated_condition', 'control_condition', 'contrast_name'])

print("Base output:", BASE_OUT)
print("Contrasts:")
display(contrasts)

# %%
# CELL 2 — Helper functions

def infer_condition(sample_name: str) -> str:
    mapped = sample_mapping_dict.get(str(sample_name), str(sample_name))
    return re.sub(r"_rep\d+$", "", mapped)


def infer_replicate(sample_name: str) -> str:
    mapped = sample_mapping_dict.get(str(sample_name), str(sample_name))
    match = re.search(r"_rep(\d+)$", mapped)
    return f"rep{match.group(1)}" if match else "repNA"


def infer_treatment(condition: str) -> str:
    condition = str(condition)
    if "+TDI" in condition:
        return "TDI"
    if "+IAG" in condition:
        return "IAG"
    return "Control"


def get_counts_chunk(adata, start: int, end: int, source: str):
    sub = adata[start:end, :]
    if source == "X":
        return sub.X
    if source.startswith("layer:"):
        layer = source.split(":", 1)[1]
        if layer not in sub.layers:
            raise KeyError(f"Layer {layer!r} not found. Available layers: {list(sub.layers.keys())}")
        return sub.layers[layer]
    raise ValueError("COUNTS_SOURCE must be 'X' or 'layer:<layer_name>'")


def as_1d_sum(x):
    if sparse.issparse(x):
        return np.asarray(x.sum(axis=0)).ravel()
    return np.asarray(x).sum(axis=0).ravel()


def bh_adjust(pvals):
    pvals = np.asarray(pvals, dtype=float)
    out = np.full_like(pvals, np.nan, dtype=float)
    valid = np.isfinite(pvals)
    p = pvals[valid]
    if p.size == 0:
        return out
    order = np.argsort(p)
    ranked = p[order]
    adj = ranked * p.size / np.arange(1, p.size + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.minimum(adj, 1.0)
    tmp = np.empty_like(adj)
    tmp[order] = adj
    out[valid] = tmp
    return out


def safe_name(x):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x)).strip("_")

# %%
# CELL 3 — Load AnnData, QC-filter cells, export pseudobulk gene counts and sample-level cNMF scores

adata = sc.read_h5ad(H5AD_PATH, backed='r')
obs = adata.obs.copy()

if SAMPLE_COL not in obs.columns:
    raise KeyError(f"{SAMPLE_COL!r} not found in adata.obs. Available: {list(obs.columns)}")
for col in ["pct_counts_mt", "pct_counts_ribo"]:
    if col not in obs.columns:
        raise KeyError(f"Required QC column {col!r} not found in adata.obs")

raw_sample = obs[SAMPLE_COL].astype(str)
obs['sample_name_de'] = raw_sample.map(lambda x: sample_mapping_dict.get(x, x))
obs['condition_de'] = obs['sample_name_de'].map(infer_condition)
obs['replicate_de'] = obs['sample_name_de'].map(infer_replicate)
obs['treatment_de'] = obs['condition_de'].map(infer_treatment)

conditions = sorted(obs['condition_de'].unique())
missing_cutoffs = sorted(set(conditions) - set(cond_pct_mt_cutoffs))
if missing_cutoffs:
    raise ValueError(f"Conditions missing QC cutoffs: {missing_cutoffs}")

mt = pd.to_numeric(obs['pct_counts_mt'], errors='coerce').to_numpy()
ribo = pd.to_numeric(obs['pct_counts_ribo'], errors='coerce').to_numpy()
cond_array = obs['condition_de'].astype(str).to_numpy()

keep = np.ones(adata.n_obs, dtype=bool)
filter_records = []
for cond in conditions:
    cond_mask = cond_array == cond
    fail_mt = cond_mask & (mt >= cond_pct_mt_cutoffs[cond])
    fail_ribo = cond_mask & (ribo <= cond_pct_ribo_cutoffs[cond])
    fail_any = fail_mt | fail_ribo
    keep &= ~fail_any
    n = int(cond_mask.sum())
    filter_records.append({
        'condition_de': cond,
        'n_cells_before': n,
        'n_fail_mt': int(fail_mt.sum()),
        'n_fail_low_ribo': int(fail_ribo.sum()),
        'n_removed_any': int(fail_any.sum()),
        'n_cells_after': int(n - fail_any.sum()),
        'fraction_removed': float(fail_any.sum() / n) if n else np.nan,
        'pct_counts_mt_cutoff': cond_pct_mt_cutoffs[cond],
        'pct_counts_ribo_cutoff': cond_pct_ribo_cutoffs[cond],
    })

filter_summary = pd.DataFrame(filter_records)
filter_summary.to_csv(BASE_OUT / 'qc_filter_summary_by_condition.tsv', sep='\t', index=False)
print(f"Cells before filter: {adata.n_obs:,}")
print(f"Cells after filter:  {keep.sum():,}")
print(f"Removed:             {(~keep).sum():,} ({(~keep).mean():.2%})")
display(filter_summary)

obs_f = obs.loc[keep].copy()

# Sample order follows the provided mapping dictionary.
mapped_order = list(sample_mapping_dict.values())
sample_order = [x for x in mapped_order if x in set(obs_f['sample_name_de'])]
sample_order += [x for x in obs_f['sample_name_de'].drop_duplicates() if x not in sample_order]
n_samples = len(sample_order)
sample_to_idx = {s: i for i, s in enumerate(sample_order)}
sample_codes_all = obs['sample_name_de'].map(sample_to_idx).fillna(-1).astype(int).to_numpy()

sample_meta_records = []
for sample in sample_order:
    mask_sample = obs_f['sample_name_de'] == sample
    cond = infer_condition(sample)
    sample_meta_records.append({
        'sample_name_de': sample,
        'condition_de': cond,
        'replicate_de': infer_replicate(sample),
        'treatment_de': infer_treatment(cond),
        'n_cells_after_filter': int(mask_sample.sum()),
        'mean_pct_counts_mt': float(pd.to_numeric(obs_f.loc[mask_sample, 'pct_counts_mt'], errors='coerce').mean()),
        'mean_pct_counts_ribo': float(pd.to_numeric(obs_f.loc[mask_sample, 'pct_counts_ribo'], errors='coerce').mean()),
    })

sample_meta = pd.DataFrame(sample_meta_records).set_index('sample_name_de')
sample_meta.to_csv(BASE_OUT / 'pseudobulk_sample_metadata.filtered.tsv', sep='\t')
contrasts.to_csv(BASE_OUT / 'matched_contrasts.tsv', sep='\t', index=False)

display(sample_meta)

# Gene IDs and annotations.
gene_id = adata.var_names.astype(str).to_numpy()
gene_annot = pd.DataFrame({'gene_id': gene_id})
if GENE_SYMBOL_COL in adata.var.columns:
    gene_annot['gene_symbol'] = adata.var[GENE_SYMBOL_COL].astype(str).to_numpy()
gene_annot.to_csv(BASE_OUT / 'gene_annotation.tsv', sep='\t', index=False)

# Pseudobulk raw counts: genes x samples output.
n_genes = adata.n_vars
counts = np.zeros((n_samples, n_genes), dtype=np.float64)

print(f"Aggregating pseudobulk counts from {COUNTS_SOURCE!r} in chunks of {CHUNK_SIZE} cells...")
for start in range(0, adata.n_obs, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, adata.n_obs)
    chunk_keep = keep[start:end]
    if not np.any(chunk_keep):
        continue
    X = get_counts_chunk(adata, start, end, COUNTS_SOURCE)
    if sparse.issparse(X):
        X = X.tocsr()[chunk_keep, :]
    else:
        X = np.asarray(X)[chunk_keep, :]
    codes = sample_codes_all[start:end][chunk_keep]
    for code in np.unique(codes):
        if code < 0 or code >= n_samples:
            continue
        counts[code, :] += as_1d_sum(X[codes == code, :])
    if start % (CHUNK_SIZE * 10) == 0:
        print(f"  processed {end:,}/{adata.n_obs:,} cells", flush=True)

max_abs_round_error = float(np.max(np.abs(counts - np.round(counts))))
print(f"Max abs count rounding error: {max_abs_round_error:.3g}")
if max_abs_round_error > 1e-4:
    print("WARNING: counts are not integer-like. Rounding for PyDESeq2; verify COUNTS_SOURCE.")

counts_int = np.rint(counts).astype(np.int64)
counts_df = pd.DataFrame(counts_int.T, index=gene_id, columns=sample_order)
counts_path = BASE_OUT / 'pseudobulk_gene_counts.filtered.tsv.gz'
counts_df.to_csv(counts_path, sep='\t', compression='gzip')
print("Wrote pseudobulk counts:", counts_path)

# Sample-level projected cNMF program scores.
print(f"Aggregating projected scores from obsm[{PROGRAM_SCORE_KEY!r}]...")
if PROGRAM_SCORE_KEY not in adata.obsm:
    raise KeyError(f"{PROGRAM_SCORE_KEY!r} not found in adata.obsm")
usage = np.asarray(adata.obsm[PROGRAM_SCORE_KEY])
projection_uns = adata.uns.get('perturbseq_cNMF_projection', {})
program_names = list(projection_uns.get('program_names', [])) if isinstance(projection_uns, dict) else []
if len(program_names) != usage.shape[1]:
    program_names = [f'P{i}' for i in range(usage.shape[1])]

score_mean = np.zeros((n_samples, usage.shape[1]), dtype=np.float64)
score_median = np.zeros((n_samples, usage.shape[1]), dtype=np.float64)
for sample, idx in sample_to_idx.items():
    cell_mask = keep & (obs['sample_name_de'].to_numpy() == sample)
    if cell_mask.sum() == 0:
        score_mean[idx, :] = np.nan
        score_median[idx, :] = np.nan
    else:
        score_mean[idx, :] = usage[cell_mask, :].mean(axis=0)
        score_median[idx, :] = np.median(usage[cell_mask, :], axis=0)

score_mean_df = pd.DataFrame(score_mean, index=sample_order, columns=program_names)
score_median_df = pd.DataFrame(score_median, index=sample_order, columns=program_names)
score_mean_df.to_csv(BASE_OUT / 'pseudobulk_projected_cNMF_mean_usage.filtered.tsv', sep='\t')
score_median_df.to_csv(BASE_OUT / 'pseudobulk_projected_cNMF_median_usage.filtered.tsv', sep='\t')
print("Wrote pseudobulk inputs to:", BASE_OUT)

adata.file.close()

# %%
# CELL 4 — Run matched pseudobulk gene DE with PyDESeq2
#
# Install first if needed:
#     python -m pip install "pydeseq2>=0.5.4"

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

DE_OUT = BASE_OUT / 'pydeseq2_gene_DE'
DE_OUT.mkdir(parents=True, exist_ok=True)

counts_gene_by_sample = pd.read_csv(BASE_OUT / 'pseudobulk_gene_counts.filtered.tsv.gz', sep='\t', index_col=0)
meta_all = pd.read_csv(BASE_OUT / 'pseudobulk_sample_metadata.filtered.tsv', sep='\t', index_col=0)
contrasts_df = pd.read_csv(BASE_OUT / 'matched_contrasts.tsv', sep='\t')
gene_annot = pd.read_csv(BASE_OUT / 'gene_annotation.tsv', sep='\t')

# Align samples and transpose to samples x genes, as expected by PyDESeq2.
common_samples = [s for s in counts_gene_by_sample.columns if s in meta_all.index]
counts_sample_by_gene = counts_gene_by_sample.loc[:, common_samples].T.round().astype(np.int64)
meta_all = meta_all.loc[common_samples, :].copy()

print("counts_sample_by_gene:", counts_sample_by_gene.shape)
print("meta_all:", meta_all.shape)

def run_one_pydeseq2_contrast(treated_condition, control_condition, contrast_name):
    print(f"\n=== {contrast_name}: {treated_condition} vs {control_condition} ===")

    samples = meta_all.index[meta_all['condition_de'].isin([control_condition, treated_condition])].tolist()
    meta_sub = meta_all.loc[samples, :].copy()

    use_paired = False
    if USE_PAIRED_REPLICATE_DESIGN:
        reps_control = set(meta_sub.loc[meta_sub['condition_de'] == control_condition, 'replicate_de'].astype(str))
        reps_treated = set(meta_sub.loc[meta_sub['condition_de'] == treated_condition, 'replicate_de'].astype(str))
        common_reps = sorted(reps_control & reps_treated)
        if len(common_reps) >= MIN_SAMPLES_PER_GROUP:
            meta_sub = meta_sub.loc[meta_sub['replicate_de'].astype(str).isin(common_reps), :].copy()
            use_paired = True

    n_control = int((meta_sub['condition_de'] == control_condition).sum())
    n_treated = int((meta_sub['condition_de'] == treated_condition).sum())
    print(f"samples control={n_control}, treated={n_treated}, paired_design={use_paired}")

    if n_control < MIN_SAMPLES_PER_GROUP or n_treated < MIN_SAMPLES_PER_GROUP:
        print("Skipping: not enough samples")
        return None, {
            'contrast_name': contrast_name,
            'treated_condition': treated_condition,
            'control_condition': control_condition,
            'n_control_samples': n_control,
            'n_treated_samples': n_treated,
            'paired_design': use_paired,
            'skipped': True,
            'skip_reason': 'not_enough_samples',
        }

    sample_ids = meta_sub.index.tolist()
    counts_sub = counts_sample_by_gene.loc[sample_ids, :].copy()

    keep_genes = ((counts_sub >= MIN_COUNT).sum(axis=0) >= MIN_SAMPLES_EXPRESSED) & (counts_sub.sum(axis=0) >= MIN_COUNT)
    print(f"genes before filter={counts_sub.shape[1]}, after filter={int(keep_genes.sum())}")
    counts_sub = counts_sub.loc[:, keep_genes]

    coldata = meta_sub.copy()
    coldata['condition_group'] = np.where(coldata['condition_de'] == treated_condition, 'treated', 'control')
    coldata['condition_group'] = pd.Categorical(coldata['condition_group'], categories=['control', 'treated'])
    coldata['replicate_de'] = pd.Categorical(coldata['replicate_de'].astype(str))

    design = '~ replicate_de + condition_group' if use_paired else '~ condition_group'
    print("design:", design)

    inference = DefaultInference(n_cpus=N_CPUS)
    dds = DeseqDataSet(
        counts=counts_sub,
        metadata=coldata,
        design=design,
        refit_cooks=False,
        size_factors_fit_type=SIZE_FACTORS_FIT_TYPE,
        inference=inference,
        quiet=QUIET_PYDESEQ2,
        low_memory=True,
    )
    dds.deseq2()

    stat_res = DeseqStats(
        dds,
        contrast=['condition_group', 'treated', 'control'],
        alpha=0.05,
        cooks_filter=True,
        independent_filter=True,
        inference=inference,
        quiet=QUIET_PYDESEQ2,
    )
    stat_res.summary()

    res_df = stat_res.results_df.copy().reset_index().rename(columns={'index': 'gene_id'})
    if 'gene_id' not in res_df.columns:
        res_df = res_df.rename(columns={res_df.columns[0]: 'gene_id'})

    res_df['contrast_name'] = contrast_name
    res_df['treated_condition'] = treated_condition
    res_df['control_condition'] = control_condition
    res_df['n_control_samples'] = n_control
    res_df['n_treated_samples'] = n_treated
    res_df['paired_design'] = use_paired
    res_df['design'] = design
    res_df['method'] = 'PyDESeq2'

    if 'gene_id' in gene_annot.columns:
        res_df = res_df.merge(gene_annot, on='gene_id', how='left', sort=False)

    preferred_cols = [
        'contrast_name', 'treated_condition', 'control_condition', 'gene_id', 'gene_symbol',
        'baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj',
        'n_control_samples', 'n_treated_samples', 'paired_design', 'design', 'method',
    ]
    extra_cols = [c for c in res_df.columns if c not in preferred_cols]
    res_df = res_df[[c for c in preferred_cols if c in res_df.columns] + extra_cols]

    sig = int(((res_df['padj'] < 0.05) & res_df['padj'].notna()).sum()) if 'padj' in res_df else 0
    sig_up = int(((res_df['padj'] < 0.05) & (res_df['log2FoldChange'] > 0) & res_df['padj'].notna()).sum())
    sig_down = int(((res_df['padj'] < 0.05) & (res_df['log2FoldChange'] < 0) & res_df['padj'].notna()).sum())

    summary = {
        'contrast_name': contrast_name,
        'treated_condition': treated_condition,
        'control_condition': control_condition,
        'n_control_samples': n_control,
        'n_treated_samples': n_treated,
        'paired_design': use_paired,
        'design': design,
        'n_genes_tested': int(res_df.shape[0]),
        'n_sig_padj_0_05': sig,
        'n_sig_up': sig_up,
        'n_sig_down': sig_down,
        'skipped': False,
        'skip_reason': '',
    }
    return res_df, summary

all_de_summaries = []
for _, row in contrasts_df.iterrows():
    res_df, summary = run_one_pydeseq2_contrast(
        treated_condition=str(row['treated_condition']),
        control_condition=str(row['control_condition']),
        contrast_name=str(row['contrast_name']),
    )
    all_de_summaries.append(summary)
    if res_df is not None:
        out_path = DE_OUT / f"PyDESeq2_{row['contrast_name']}.tsv"
        res_df.to_csv(out_path, sep='\t', index=False)
        print("wrote", out_path)

de_summary_df = pd.DataFrame(all_de_summaries)
de_summary_df.to_csv(DE_OUT / 'PyDESeq2_contrast_summary.tsv', sep='\t', index=False)
display(de_summary_df)

# %%
# CELL 5 — Matched treatment-vs-control tests for projected cNMF program scores

PROGRAM_OUT = BASE_OUT / 'program_score_DE'
PROGRAM_OUT.mkdir(parents=True, exist_ok=True)

scores = pd.read_csv(BASE_OUT / 'pseudobulk_projected_cNMF_mean_usage.filtered.tsv', sep='\t', index_col=0)
meta = pd.read_csv(BASE_OUT / 'pseudobulk_sample_metadata.filtered.tsv', sep='\t', index_col=0)
contrasts_df = pd.read_csv(BASE_OUT / 'matched_contrasts.tsv', sep='\t')

scores = scores.loc[meta.index]
program_names = list(scores.columns)

records = []
paired_records = []
EPS = 1e-8

for _, contrast in contrasts_df.iterrows():
    cname = contrast['contrast_name']
    trt = contrast['treated_condition']
    ctrl = contrast['control_condition']

    ctrl_samples = meta.index[meta['condition_de'] == ctrl].tolist()
    trt_samples = meta.index[meta['condition_de'] == trt].tolist()
    reps_ctrl = set(meta.loc[ctrl_samples, 'replicate_de'])
    reps_trt = set(meta.loc[trt_samples, 'replicate_de'])
    common_reps = sorted(reps_ctrl & reps_trt)
    use_paired = len(common_reps) >= 2

    for program in program_names:
        ctrl_vals = scores.loc[ctrl_samples, program].astype(float)
        trt_vals = scores.loc[trt_samples, program].astype(float)

        mean_ctrl = float(ctrl_vals.mean())
        mean_trt = float(trt_vals.mean())
        diff = mean_trt - mean_ctrl
        log2_ratio = float(np.log2((mean_trt + EPS) / (mean_ctrl + EPS)))

        if use_paired:
            paired_diffs = []
            for rep in common_reps:
                c_s = meta.index[(meta['condition_de'] == ctrl) & (meta['replicate_de'] == rep)].tolist()
                t_s = meta.index[(meta['condition_de'] == trt) & (meta['replicate_de'] == rep)].tolist()
                if len(c_s) == 1 and len(t_s) == 1:
                    d = float(scores.loc[t_s[0], program] - scores.loc[c_s[0], program])
                    paired_diffs.append(d)
                    paired_records.append({
                        'contrast_name': cname,
                        'program': program,
                        'replicate_de': rep,
                        'control_sample': c_s[0],
                        'treated_sample': t_s[0],
                        'control_value': float(scores.loc[c_s[0], program]),
                        'treated_value': float(scores.loc[t_s[0], program]),
                        'paired_diff': d,
                    })
            paired_diffs = np.asarray(paired_diffs)
            if paired_diffs.size >= 2 and np.std(paired_diffs) > 0:
                stat, pval = stats.ttest_1samp(paired_diffs, popmean=0.0)
            else:
                stat, pval = np.nan, np.nan
            test = 'paired_ttest_on_replicate_differences'
            n_pairs = int(paired_diffs.size)
        else:
            if len(ctrl_vals) >= 2 and len(trt_vals) >= 2:
                stat, pval = stats.ttest_ind(trt_vals, ctrl_vals, equal_var=False, nan_policy='omit')
            else:
                stat, pval = np.nan, np.nan
            test = 'welch_ttest'
            n_pairs = 0

        records.append({
            'contrast_name': cname,
            'treated_condition': trt,
            'control_condition': ctrl,
            'program': program,
            'mean_control': mean_ctrl,
            'mean_treated': mean_trt,
            'diff_treated_minus_control': diff,
            'log2_ratio_treated_vs_control': log2_ratio,
            'statistic': stat,
            'pvalue': pval,
            'test': test,
            'n_control_samples': len(ctrl_samples),
            'n_treated_samples': len(trt_samples),
            'n_matched_pairs': n_pairs,
        })

program_res = pd.DataFrame(records)
program_res['padj_bh_within_contrast'] = np.nan
for cname, idx in program_res.groupby('contrast_name').groups.items():
    program_res.loc[idx, 'padj_bh_within_contrast'] = bh_adjust(program_res.loc[idx, 'pvalue'].to_numpy())

program_res = program_res.sort_values(['contrast_name', 'padj_bh_within_contrast', 'pvalue', 'program'])
program_res.to_csv(PROGRAM_OUT / 'projected_cNMF_program_score_matched_contrasts.tsv', sep='\t', index=False)

if paired_records:
    pd.DataFrame(paired_records).to_csv(PROGRAM_OUT / 'projected_cNMF_program_score_paired_replicate_differences.tsv', sep='\t', index=False)

key_program_res = program_res[program_res['program'].isin(KEY_PROGRAMS)].copy()
key_program_res.to_csv(PROGRAM_OUT / 'key_programs_P15_P17_P34_matched_contrasts.tsv', sep='\t', index=False)
display(key_program_res)

# Plot key programs.
for program in KEY_PROGRAMS:
    if program not in program_names:
        print(f"Skipping {program}: not in program score table")
        continue
    prog_res = program_res[program_res['program'] == program].set_index('contrast_name')
    fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(prog_res)), 3.5))
    x = np.arange(len(prog_res))
    ax.axhline(0, linewidth=0.8)
    ax.bar(x, prog_res['diff_treated_minus_control'].to_numpy())
    ax.set_xticks(x)
    ax.set_xticklabels(prog_res.index.tolist(), rotation=90)
    ax.set_ylabel(f'{program} mean usage difference\n(treated - control)')
    ax.set_title(f'{program} matched treatment effects')
    fig.tight_layout()
    fig.savefig(PROGRAM_OUT / f'{program}_matched_treatment_effects.svg')
    fig.savefig(PROGRAM_OUT / f'{program}_matched_treatment_effects.png')
    plt.show()
    plt.close(fig)

print('Wrote program-score contrast outputs to:', PROGRAM_OUT)

# %%
# CELL 6 — Optional: cNMF program enrichment from PyDESeq2 gene log2FC
#
# This checks whether genes with high weights in a perturb-seq cNMF program tend
# to increase or decrease in each DE contrast.

ENRICH_OUT = BASE_OUT / 'program_enrichment_from_PyDESeq2_gene_DE'
ENRICH_OUT.mkdir(parents=True, exist_ok=True)
TOP_N = 200

spectra = pd.read_csv(SPECTRA_TPM_PATH, sep='\t', index_col=0)
spectra.index = [f'P{int(x) - 1}' for x in spectra.index.astype(str)]  # source rows 1..50 -> P0..P49
spectra.columns = spectra.columns.astype(str)
spectra = spectra.apply(pd.to_numeric, errors='coerce').fillna(0)
spectra[spectra < 0] = 0

result_files = sorted(DE_OUT.glob('PyDESeq2_*.tsv'))
if not result_files:
    raise FileNotFoundError(f"No PyDESeq2_*.tsv files found in {DE_OUT}")

all_records = []
for path in result_files:
    if path.name == 'PyDESeq2_contrast_summary.tsv':
        continue
    de = pd.read_csv(path, sep='\t')
    contrast = path.stem.replace('PyDESeq2_', '')
    print('Reading:', path.name)

    if 'gene_symbol' in de.columns:
        gene_col = 'gene_symbol'
    elif 'gene_id' in de.columns:
        gene_col = 'gene_id'
    else:
        print(f'Skipping {path.name}: no gene_symbol or gene_id column')
        continue

    if 'log2FoldChange' not in de.columns:
        print(f'Skipping {path.name}: no log2FoldChange column')
        continue

    de = de.dropna(subset=['log2FoldChange']).copy()
    de[gene_col] = de[gene_col].astype(str)
    de = de.drop_duplicates(gene_col, keep='first').set_index(gene_col)

    common = de.index.intersection(spectra.columns)
    print(f"  shared genes with spectra: {len(common):,}")
    if len(common) == 0:
        continue

    lfc = de.loc[common, 'log2FoldChange'].astype(float)
    stat = de.loc[common, 'stat'].astype(float) if 'stat' in de.columns else lfc

    for program in spectra.index:
        weights = spectra.loc[program, common].astype(float)
        weight_sum = weights.sum()
        if weight_sum <= 0:
            continue

        weighted_lfc = float(np.sum(weights * lfc) / weight_sum)
        weighted_stat = float(np.sum(weights * stat) / weight_sum)

        top_genes = weights.sort_values(ascending=False).head(TOP_N).index
        top_lfc_mean = float(lfc.loc[top_genes].mean())
        top_stat_mean = float(stat.loc[top_genes].mean())
        rho, pval = stats.spearmanr(weights.to_numpy(), lfc.to_numpy(), nan_policy='omit')

        all_records.append({
            'method': 'PyDESeq2',
            'contrast_name': contrast,
            'program': program,
            'n_shared_genes': len(common),
            'weighted_mean_log2FC_by_spectra': weighted_lfc,
            'weighted_mean_stat_by_spectra': weighted_stat,
            f'top_{TOP_N}_genes_mean_log2FC': top_lfc_mean,
            f'top_{TOP_N}_genes_mean_stat': top_stat_mean,
            'spearman_weight_vs_log2FC': rho,
            'spearman_pvalue': pval,
            f'top_{TOP_N}_genes': ','.join(top_genes.tolist()),
        })

enrich_res = pd.DataFrame(all_records)
if enrich_res.empty:
    raise RuntimeError('No enrichment records were generated')

enrich_res['spearman_padj_bh_within_contrast'] = np.nan
for contrast, idx in enrich_res.groupby('contrast_name').groups.items():
    enrich_res.loc[idx, 'spearman_padj_bh_within_contrast'] = bh_adjust(enrich_res.loc[idx, 'spearman_pvalue'].to_numpy())

enrich_res = enrich_res.sort_values(['contrast_name', 'weighted_mean_log2FC_by_spectra'], ascending=[True, False])
enrich_res.to_csv(ENRICH_OUT / 'program_enrichment_from_PyDESeq2_log2FC.tsv', sep='\t', index=False)

key_enrich = enrich_res[enrich_res['program'].isin(KEY_PROGRAMS)].copy()
key_enrich.to_csv(ENRICH_OUT / 'key_programs_P15_P17_P34_enrichment_from_PyDESeq2.tsv', sep='\t', index=False)

display(key_enrich)
print('Wrote enrichment outputs to:', ENRICH_OUT)


In [ ]:
# %% [markdown]
# # cNMF program overlap and Hippo KD concordance analysis
#
# This notebook uses the cNMF Excel workbook:
#
# - `Program Loadings`: top 300 loading genes per cNMF program
# - `Perturbation Association 1`: effect size of each perturbation/KD on each program on each day
#
# It performs two analyses:
#
# 1. **Top program gene overlap with small-molecule DE genes**
# 2. **Program-level concordance between small-molecule effects and perturb-seq Hippo KD effects**
#
# This does not require the large h5ad/h5mu files.

# %%
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact, spearmanr, pearsonr
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# ---- Paths ----
# On Sherlock, change BASE_DIR if needed.
BASE_DIR = Path("/scratch/users/edtng/20260325_Hippo_timecourse/perturbseq_cNMF_matched_DE")

# Use uploaded/local paths if running this notebook beside the uploaded files.
LOCAL_DIR = Path("/mnt/data")

CNMF_XLSX = Path("/mnt/data/cNMF_50_0_2_032526_6co.xlsx")
if not CNMF_XLSX.exists():
    CNMF_XLSX = Path("/scratch/users/edtng/20260325_Hippo_timecourse/cNMF_50_0_2_032526_6co.xlsx")

# Directory containing PyDESeq2_*.tsv files.
DE_DIR = LOCAL_DIR
if not list(DE_DIR.glob("PyDESeq2_*.tsv")):
    DE_DIR = BASE_DIR / "pydeseq2_gene_DE"

# Small-molecule projected program score contrasts.
PROGRAM_SCORE_CONTRASTS = LOCAL_DIR / "projected_cNMF_program_score_matched_contrasts.tsv"
if not PROGRAM_SCORE_CONTRASTS.exists():
    PROGRAM_SCORE_CONTRASTS = BASE_DIR / "program_score_DE" / "projected_cNMF_program_score_matched_contrasts.tsv"

OUT_DIR = Path("/mnt/data/hippo_program_overlap_concordance_outputs")
if not Path("/mnt/data").exists():
    OUT_DIR = BASE_DIR / "program_overlap_concordance"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("CNMF_XLSX:", CNMF_XLSX, CNMF_XLSX.exists())
print("DE_DIR:", DE_DIR, len(list(DE_DIR.glob("PyDESeq2_*.tsv"))))
print("PROGRAM_SCORE_CONTRASTS:", PROGRAM_SCORE_CONTRASTS, PROGRAM_SCORE_CONTRASTS.exists())
print("OUT_DIR:", OUT_DIR)

# ---- Analysis parameters ----
DE_PADJ_CUTOFF = 0.05
DE_ABS_LFC_CUTOFF = 0.0
TOP_N_PROGRAM_GENES = 300

# Hippo-ish KD targets found in the workbook. Edit as needed.
# I would start conservatively with core/regulatory Hippo pathway genes that are present.
HIPPO_KD_TARGETS = ["LATS2", "NF2", "SAV1", "TAOK1", "WWC3", "AMOTL2"]

# Effect column in Perturbation Association 1.
# approx_log2FC is easiest to interpret; beta_hat is also available.
PERTURBSEQ_PROGRAM_EFFECT_COL = "approx_log2FC"

KEY_PROGRAMS = ["P15", "P17", "P34"]

# %% [markdown]
# ## Helper functions

# %%
def bh_adjust(pvals):
    """Benjamini-Hochberg FDR correction. Returns q-values in original order."""
    pvals = np.asarray(pvals, dtype=float)
    out = np.full_like(pvals, np.nan, dtype=float)
    valid = np.isfinite(pvals)
    if valid.sum() == 0:
        return out
    p = pvals[valid]
    order = np.argsort(p)
    ranked = p[order]
    n = len(ranked)
    q = ranked * n / (np.arange(n) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.minimum(q, 1.0)
    temp = np.empty_like(q)
    temp[order] = q
    out[valid] = temp
    return out


def normalize_program_name(x):
    """Return zero-based P# program label."""
    if isinstance(x, str) and x.startswith("P"):
        return x
    return f"P{int(float(x))}"


def contrast_to_stage_day(contrast_name):
    """Map small-molecule contrast names to perturb-seq association Sample day D0/D1/D2/D3."""
    cn = str(contrast_name)
    if "D0" in cn:
        return "D0"
    if "D1_MPS" in cn:
        return "D1"
    if "D1.33_LM" in cn or "D1.67_LM" in cn or "D2_LM" in cn:
        # LM-stage comparisons correspond most closely to D2 in perturb-seq association table.
        return "D2"
    if "D3_AE" in cn:
        return "D3"
    return None


def read_deseq_table(path):
    df = pd.read_csv(path, sep="\t")
    if "contrast_name" not in df.columns:
        name = Path(path).stem.replace("PyDESeq2_", "")
        df["contrast_name"] = name
    if "gene_symbol" in df.columns:
        df["gene_for_overlap"] = df["gene_symbol"].astype(str)
    elif "gene" in df.columns:
        df["gene_for_overlap"] = df["gene"].astype(str)
    elif "gene_id" in df.columns:
        df["gene_for_overlap"] = df["gene_id"].astype(str)
    else:
        raise ValueError(f"Could not find gene column in {path}. Columns: {list(df.columns)}")
    return df

# %% [markdown]
# ## Load cNMF workbook tabs

# %%
program_loadings = pd.read_excel(CNMF_XLSX, sheet_name="Program Loadings")
assoc = pd.read_excel(CNMF_XLSX, sheet_name="Perturbation Association 1")

print("Program Loadings shape:", program_loadings.shape)
print(program_loadings.head())
print("\nPerturbation Association 1 shape:", assoc.shape)
print(assoc.head())

# Clean columns
program_loadings["Program"] = program_loadings["Program"].astype(int)
program_loadings["Rank"] = program_loadings["Rank"].astype(int)
program_loadings["Gene"] = program_loadings["Gene"].astype(str)
program_loadings["program"] = program_loadings["Program"].map(lambda x: f"P{x}")

assoc["program"] = assoc["program_name"].map(normalize_program_name)
assoc["target_name"] = assoc["target_name"].astype(str)
assoc["Sample"] = assoc["Sample"].astype(str)

# Show Hippo-like targets found in workbook to help edit HIPPO_KD_TARGETS.
hippo_keywords = ["LATS", "STK3", "STK4", "SAV", "NF2", "MOB", "YAP", "WWTR", "TEAD", "AMOT", "TAOK", "FAT", "DCHS", "FRMD", "WWC"]
all_targets = sorted(assoc["target_name"].dropna().unique().tolist())
hippo_like_targets = [t for t in all_targets if any(k.lower() in t.lower() for k in hippo_keywords)]
print("\nHippo-like targets found in workbook:")
print(hippo_like_targets)

available_hippo_targets = [t for t in HIPPO_KD_TARGETS if t in set(all_targets)]
missing_hippo_targets = [t for t in HIPPO_KD_TARGETS if t not in set(all_targets)]
print("\nUsing HIPPO_KD_TARGETS available in workbook:", available_hippo_targets)
print("Missing from workbook:", missing_hippo_targets)
if not available_hippo_targets:
    raise ValueError("No HIPPO_KD_TARGETS were found in association table. Edit HIPPO_KD_TARGETS.")

# Top N loading gene sets
program_top_genes = {}
for program, sub in program_loadings.sort_values(["Program", "Rank"]).groupby("program"):
    program_top_genes[program] = sub.head(TOP_N_PROGRAM_GENES)["Gene"].astype(str).tolist()

print("n programs with top genes:", len(program_top_genes))
print("P15 top genes:", program_top_genes.get("P15", [])[:20])

# %% [markdown]
# ## Analysis 1: overlap between small-molecule DE genes and top cNMF loading genes
#
# This is descriptive/supportive. It asks whether top loading genes from a program are overrepresented among genes up- or down-regulated in each small molecule contrast.

# %%
de_files = sorted([p for p in DE_DIR.glob("PyDESeq2_*.tsv") if "contrast_summary" not in p.name])
print("DE files:", len(de_files))
for p in de_files[:5]:
    print(" ", p.name)

all_overlap_records = []

for de_path in de_files:
    de = read_deseq_table(de_path)
    contrast_name = str(de["contrast_name"].iloc[0])

    de = de.dropna(subset=["gene_for_overlap", "log2FoldChange", "padj"]).copy()
    de["gene_for_overlap"] = de["gene_for_overlap"].astype(str)

    background = set(de["gene_for_overlap"])
    de_up = set(de.loc[(de["padj"] < DE_PADJ_CUTOFF) & (de["log2FoldChange"] > DE_ABS_LFC_CUTOFF), "gene_for_overlap"])
    de_down = set(de.loc[(de["padj"] < DE_PADJ_CUTOFF) & (de["log2FoldChange"] < -DE_ABS_LFC_CUTOFF), "gene_for_overlap"])

    for program, genes in program_top_genes.items():
        prog_set = set(genes) & background
        for direction, de_set in [("up", de_up), ("down", de_down)]:
            overlap = prog_set & de_set
            a = len(overlap)
            b = len(prog_set) - a
            c = len(de_set) - a
            d = len(background) - a - b - c
            odds, pvalue = fisher_exact([[a, b], [c, d]], alternative="greater")
            all_overlap_records.append({
                "contrast_name": contrast_name,
                "program": program,
                "direction": direction,
                "n_background": len(background),
                "n_program_top_genes_in_background": len(prog_set),
                "n_de_genes": len(de_set),
                "n_overlap": a,
                "odds_ratio": odds,
                "pvalue": pvalue,
                "overlap_genes": ";".join(sorted(overlap)),
            })

overlap_df = pd.DataFrame(all_overlap_records)
overlap_df["padj_bh_all"] = bh_adjust(overlap_df["pvalue"].to_numpy())
overlap_df["padj_bh_within_contrast"] = np.nan
for contrast, idx in overlap_df.groupby("contrast_name").groups.items():
    overlap_df.loc[idx, "padj_bh_within_contrast"] = bh_adjust(overlap_df.loc[idx, "pvalue"].to_numpy())

overlap_df = overlap_df.sort_values(["contrast_name", "padj_bh_within_contrast", "pvalue"])
out_path = OUT_DIR / "program_top300_gene_overlap_with_small_molecule_DE.tsv"
overlap_df.to_csv(out_path, sep="\t", index=False)
print("Saved:", out_path)

display(overlap_df.query("program in @KEY_PROGRAMS").sort_values(["contrast_name", "program", "direction"]).head(50))

# %% [markdown]
# ## Analysis 2: program-level concordance between small molecule effects and perturb-seq Hippo KD effects
#
# This uses the `Perturbation Association 1` tab. It compares:
#
# - small-molecule projected cNMF program effects, e.g. TDI vs control
# - perturb-seq Hippo KD effect vector across all 50 programs
#
# For TDI, we expect positive concordance with Hippo KD if TDI mimics Hippo inhibition. For IAG, we may expect weaker or opposite concordance.

# %%
sm_prog = pd.read_csv(PROGRAM_SCORE_CONTRASTS, sep="\t")
sm_prog["program"] = sm_prog["program"].astype(str)
sm_prog["stage_day_for_perturbseq"] = sm_prog["contrast_name"].map(contrast_to_stage_day)

print(sm_prog.shape)
print(sm_prog.head())
print("Stage/day mapping:")
display(sm_prog[["contrast_name", "stage_day_for_perturbseq"]].drop_duplicates().sort_values("contrast_name"))

# Average perturb-seq Hippo KD effects across selected Hippo targets for each Sample x program.
hippo_assoc = assoc.loc[assoc["target_name"].isin(available_hippo_targets)].copy()
hippo_assoc[PERTURBSEQ_PROGRAM_EFFECT_COL] = pd.to_numeric(hippo_assoc[PERTURBSEQ_PROGRAM_EFFECT_COL], errors="coerce")

hippo_avg = (
    hippo_assoc
    .groupby(["Sample", "program"], as_index=False)
    .agg(
        hippo_kd_mean_effect=(PERTURBSEQ_PROGRAM_EFFECT_COL, "mean"),
        hippo_kd_median_effect=(PERTURBSEQ_PROGRAM_EFFECT_COL, "median"),
        n_hippo_targets=("target_name", "nunique"),
        hippo_targets=("target_name", lambda x: ";".join(sorted(set(map(str, x))))),
    )
)

print("Hippo KD averaged effect table:")
display(hippo_avg.head())

# %%
concordance_records = []
merged_records = []

for contrast_name, sub in sm_prog.groupby("contrast_name"):
    stage_day = sub["stage_day_for_perturbseq"].iloc[0]
    if stage_day is None or pd.isna(stage_day):
        continue
    kd_sub = hippo_avg.loc[hippo_avg["Sample"] == stage_day].copy()
    merged = sub.merge(kd_sub, on="program", how="inner")
    if merged.empty:
        continue

    x = merged["hippo_kd_mean_effect"].to_numpy(dtype=float)
    y = merged["diff_treated_minus_control"].to_numpy(dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]
    merged = merged.loc[valid].copy()

    pearson_r, pearson_p = pearsonr(x, y) if len(x) > 2 else (np.nan, np.nan)
    spearman_r, spearman_p = spearmanr(x, y) if len(x) > 2 else (np.nan, np.nan)

    concordance_records.append({
        "contrast_name": contrast_name,
        "treated_condition": sub["treated_condition"].iloc[0],
        "control_condition": sub["control_condition"].iloc[0],
        "stage_day_for_perturbseq": stage_day,
        "small_molecule_effect_col": "diff_treated_minus_control",
        "perturbseq_effect_col": PERTURBSEQ_PROGRAM_EFFECT_COL,
        "hippo_targets_used": ";".join(available_hippo_targets),
        "n_programs": len(x),
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_r": spearman_r,
        "spearman_p": spearman_p,
    })
    merged["contrast_name"] = contrast_name
    merged_records.append(merged)

concordance_df = pd.DataFrame(concordance_records).sort_values("contrast_name")
concordance_df["spearman_padj_bh"] = bh_adjust(concordance_df["spearman_p"].to_numpy())
concordance_df["pearson_padj_bh"] = bh_adjust(concordance_df["pearson_p"].to_numpy())

program_level_merged_df = pd.concat(merged_records, ignore_index=True) if merged_records else pd.DataFrame()

out1 = OUT_DIR / "small_molecule_vs_perturbseq_HippoKD_program_concordance.tsv"
out2 = OUT_DIR / "small_molecule_vs_perturbseq_HippoKD_program_level_values.tsv"
concordance_df.to_csv(out1, sep="\t", index=False)
program_level_merged_df.to_csv(out2, sep="\t", index=False)

print("Saved:", out1)
print("Saved:", out2)
display(concordance_df)

# Key program details
key_program_details = program_level_merged_df.loc[program_level_merged_df["program"].isin(KEY_PROGRAMS)].copy()
key_path = OUT_DIR / "small_molecule_vs_perturbseq_HippoKD_key_programs.tsv"
key_program_details.to_csv(key_path, sep="\t", index=False)
print("Saved:", key_path)
display(key_program_details[["contrast_name", "program", "diff_treated_minus_control", "hippo_kd_mean_effect", "n_hippo_targets", "hippo_targets"]].head(100))

# %% [markdown]
# ## Quick plots: concordance across all programs and key programs

# %%
# Bar plot of Spearman correlations across all programs
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = concordance_df.copy()
plot_df = plot_df.sort_values("spearman_r")
colors = ["#C2410C" if "TDI" in x else "#1D4ED8" for x in plot_df["contrast_name"]]
ax.barh(plot_df["contrast_name"], plot_df["spearman_r"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Spearman correlation across programs\nsmall molecule effect vs perturb-seq Hippo KD effect")
ax.set_ylabel("Contrast")
ax.set_title("Program-level concordance with perturb-seq Hippo KD")
fig.tight_layout()
fig.savefig(OUT_DIR / "program_level_concordance_HippoKD_spearman.svg")
fig.savefig(OUT_DIR / "program_level_concordance_HippoKD_spearman.pdf")
fig.savefig(OUT_DIR / "program_level_concordance_HippoKD_spearman.png", dpi=300)
plt.show()

# %%
# Scatter plots for selected contrasts
SELECTED_CONTRASTS = [
    "TDI_at_D2_LM",
    "IAG_at_D2_LM",
    "TDI_at_D3_AE",
    "IAG_at_D3_AE",
    "TDI_D2_to_D3_vs_D3_AE",
    "IAG_D2_to_D3_vs_D3_AE",
]

n = len(SELECTED_CONTRASTS)
fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=False, sharey=False)
axes = axes.ravel()

for ax, contrast in zip(axes, SELECTED_CONTRASTS):
    sub = program_level_merged_df.loc[program_level_merged_df["contrast_name"] == contrast].copy()
    if sub.empty:
        ax.axis("off")
        ax.set_title(contrast + "\nmissing")
        continue

    ax.scatter(sub["hippo_kd_mean_effect"], sub["diff_treated_minus_control"], s=18, alpha=0.7)
    for _, row in sub.loc[sub["program"].isin(KEY_PROGRAMS)].iterrows():
        ax.scatter(row["hippo_kd_mean_effect"], row["diff_treated_minus_control"], s=45, edgecolor="black")
        ax.text(row["hippo_kd_mean_effect"], row["diff_treated_minus_control"], row["program"], fontsize=8)
    ax.axhline(0, color="grey", linewidth=0.7)
    ax.axvline(0, color="grey", linewidth=0.7)
    corr = concordance_df.loc[concordance_df["contrast_name"] == contrast, "spearman_r"]
    corr_txt = f"rho={corr.iloc[0]:.2f}" if len(corr) else "rho=NA"
    ax.set_title(f"{contrast}\n{corr_txt}", fontsize=8)
    ax.set_xlabel("Perturb-seq Hippo KD program effect")
    ax.set_ylabel("Small molecule program effect")

fig.tight_layout()
fig.savefig(OUT_DIR / "program_level_concordance_HippoKD_scatter_selected.svg")
fig.savefig(OUT_DIR / "program_level_concordance_HippoKD_scatter_selected.pdf")
fig.savefig(OUT_DIR / "program_level_concordance_HippoKD_scatter_selected.png", dpi=300)
plt.show()

print("All outputs saved to", OUT_DIR)


#### Redo with CLR for small molecule data as well

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import spearmanr, pearsonr, ttest_rel

BASE = Path("/scratch/users/edtng/20260325_Hippo_timecourse")
PROJ_DIR = BASE / "perturbseq_cNMF_projection"
OUT_DIR = BASE / "perturbseq_cNMF_matched_DE" / "program_score_DE_clr"
OUT_DIR.mkdir(parents=True, exist_ok=True)

H5AD_PATH = PROJ_DIR / "Hippo_timecourse_combined.projected_perturbseq_cNMF_50_dt_0_2.zero_based.h5ad"

XLSX_PATH = Path("/scratch/users/edtng/20260325_Hippo_timecourse/cNMF_50_0_2_032526_6co.xlsx")
# On Sherlock, change to wherever you store the workbook:
# XLSX_PATH = Path("/scratch/users/edtng/.../cNMF_50_0_2_032526_6co.xlsx")

PROGRAM_EFFECT_SHEET = "Perturbation Association 1"

# Use approx_log2FC if you want log2-scale concordance.
PERTURBSEQ_EFFECT_COL = "approx_log2FC"
LOG_BASE = 2

# Or use this pair if you want natural-log beta scale:
# PERTURBSEQ_EFFECT_COL = "beta_hat"
# LOG_BASE = "natural"

HIPPO_KD_TARGETS = ["LATS2", "NF2", "SAV1", "TAOK1", "WWC3", "AMOTL2"]

SCORE_KEY = "X_perturbseq_cNMF_usage"
SAMPLE_COL = "sample_name"

KEY_PROGRAMS = ["P15", "P17", "P27", "P34"]

In [ ]:
def clr_transform_usage(usage, eps=None, eps_quantile=1e-4, log_base=2):
    """
    usage: cells x programs, rows should sum to 1.
    Returns CLR-transformed usage with same shape.
    """
    usage = np.asarray(usage, dtype=np.float64)

    row_sums = usage.sum(axis=1, keepdims=True)
    usage = np.divide(
        usage,
        row_sums,
        out=np.zeros_like(usage),
        where=row_sums > 0,
    )

    if eps is None:
        positive = usage[usage > 0]
        if positive.size == 0:
            raise ValueError("No positive usage values found.")
        eps = float(np.quantile(positive, eps_quantile))

    u = np.maximum(usage, eps)
    u = u / u.sum(axis=1, keepdims=True)

    if log_base == 2:
        log_u = np.log2(u)
    elif log_base == "natural":
        log_u = np.log(u)
    else:
        raise ValueError("log_base must be 2 or 'natural'.")

    clr = log_u - log_u.mean(axis=1, keepdims=True)
    return clr.astype(np.float32), eps


def parse_sample_name(sample):
    """
    Example:
      D2_LM+TDI_rep1 -> condition D2_LM+TDI, replicate rep1
    """
    sample = str(sample)
    m = re.match(r"(.+)_rep(\d+)$", sample)
    if m is None:
        raise ValueError(f"Could not parse replicate from sample_name={sample!r}")
    return m.group(1), f"rep{m.group(2)}"


def make_contrast_table():
    rows = []
    for base_cond in ["D1_MPS", "D1.33_LM", "D1.67_LM", "D2_LM", "D3_AE"]:
        for treatment in ["TDI", "IAG"]:
            treated = f"{base_cond}+{treatment}"
            rows.append({
                "contrast": treated,
                "control_condition": base_cond,
                "treated_condition": treated,
                "treatment": treatment,
            })

    rows += [
        {
            "contrast": "D2_LM+TDI->D3_AE+TDI",
            "control_condition": "D3_AE",
            "treated_condition": "D2_LM+TDI->D3_AE+TDI",
            "treatment": "TDI",
        },
        {
            "contrast": "D2_LM+IAG->D3_AE+IAG",
            "control_condition": "D3_AE",
            "treated_condition": "D2_LM+IAG->D3_AE+IAG",
            "treatment": "IAG",
        },
    ]
    return pd.DataFrame(rows)


contrast_df = make_contrast_table()
display(contrast_df)

In [ ]:
adata = sc.read_h5ad(H5AD_PATH, backed="r")

usage = np.asarray(adata.obsm[SCORE_KEY])
obs = adata.obs.copy()

projection_meta = adata.uns.get("perturbseq_cNMF_projection", {})
program_names = [str(x) for x in projection_meta.get("program_names", [])]
if len(program_names) != usage.shape[1]:
    program_names = [f"P{i}" for i in range(usage.shape[1])]

print("usage shape:", usage.shape)
print("programs:", program_names[:5], "...", program_names[-5:])

clr_usage, clr_eps = clr_transform_usage(
    usage,
    eps=None,
    eps_quantile=1e-4,
    log_base=LOG_BASE,
)

print("CLR eps:", clr_eps)
print("CLR shape:", clr_usage.shape)
print("CLR row mean range:", clr_usage.mean(axis=1).min(), clr_usage.mean(axis=1).max())

# Build cell-level CLR dataframe only for program scores.
clr_df = pd.DataFrame(
    clr_usage,
    index=obs.index.astype(str),
    columns=program_names,
)

sample_meta = obs[[SAMPLE_COL]].copy()
sample_meta[SAMPLE_COL] = sample_meta[SAMPLE_COL].astype(str)
parsed = sample_meta[SAMPLE_COL].map(parse_sample_name)
sample_meta["condition"] = [x[0] for x in parsed]
sample_meta["replicate"] = [x[1] for x in parsed]

clr_df[SAMPLE_COL] = sample_meta[SAMPLE_COL].to_numpy()
clr_df["condition"] = sample_meta["condition"].to_numpy()
clr_df["replicate"] = sample_meta["replicate"].to_numpy()

sample_clr_mean = (
    clr_df
    .groupby(SAMPLE_COL, observed=True)[program_names]
    .mean()
)

sample_clr_meta = (
    sample_meta
    .drop_duplicates(SAMPLE_COL)
    .set_index(SAMPLE_COL)
    .loc[sample_clr_mean.index]
)

display(sample_clr_meta.head())
display(sample_clr_mean.head())

sample_clr_mean.to_csv(OUT_DIR / "sample_mean_CLR_projected_cNMF_usage.tsv", sep="\t")
sample_clr_meta.to_csv(OUT_DIR / "sample_mean_CLR_projected_cNMF_metadata.tsv", sep="\t")

adata.file.close()

In [ ]:
records = []
paired_records = []

for _, contrast in contrast_df.iterrows():
    control_cond = contrast["control_condition"]
    treated_cond = contrast["treated_condition"]
    contrast_name = contrast["contrast"]

    control_samples = sample_clr_meta.index[sample_clr_meta["condition"] == control_cond].tolist()
    treated_samples = sample_clr_meta.index[sample_clr_meta["condition"] == treated_cond].tolist()

    control_by_rep = {
        sample_clr_meta.loc[s, "replicate"]: s
        for s in control_samples
    }
    treated_by_rep = {
        sample_clr_meta.loc[s, "replicate"]: s
        for s in treated_samples
    }

    matched_reps = sorted(set(control_by_rep).intersection(treated_by_rep))

    if len(matched_reps) == 0:
        print(f"Skipping {contrast_name}: no matched replicates.")
        continue

    for program in program_names:
        diffs = []
        control_vals = []
        treated_vals = []

        for rep in matched_reps:
            c_sample = control_by_rep[rep]
            t_sample = treated_by_rep[rep]

            c_val = float(sample_clr_mean.loc[c_sample, program])
            t_val = float(sample_clr_mean.loc[t_sample, program])
            diff = t_val - c_val

            control_vals.append(c_val)
            treated_vals.append(t_val)
            diffs.append(diff)

            paired_records.append({
                "contrast": contrast_name,
                "treatment": contrast["treatment"],
                "control_condition": control_cond,
                "treated_condition": treated_cond,
                "program": program,
                "replicate": rep,
                "control_sample": c_sample,
                "treated_sample": t_sample,
                "control_clr": c_val,
                "treated_clr": t_val,
                "paired_diff_clr": diff,
            })

        diffs = np.asarray(diffs, dtype=float)

        if len(diffs) >= 2:
            stat, pval = ttest_rel(treated_vals, control_vals)
        else:
            stat, pval = np.nan, np.nan

        records.append({
            "contrast": contrast_name,
            "treatment": contrast["treatment"],
            "control_condition": control_cond,
            "treated_condition": treated_cond,
            "program": program,
            "n_matched_pairs": len(matched_reps),
            "mean_control_clr": np.mean(control_vals),
            "mean_treated_clr": np.mean(treated_vals),
            "mean_diff_clr": np.mean(diffs),
            "sd_diff_clr": np.std(diffs, ddof=1) if len(diffs) > 1 else np.nan,
            "paired_t_stat": stat,
            "paired_t_pvalue": pval,
            "n_positive_diffs": int((diffs > 0).sum()),
            "n_negative_diffs": int((diffs < 0).sum()),
        })

smallmol_clr_effects = pd.DataFrame(records)
smallmol_clr_paired = pd.DataFrame(paired_records)

smallmol_clr_effects.to_csv(
    OUT_DIR / "small_molecule_projected_cNMF_CLR_matched_effects.tsv",
    sep="\t",
    index=False,
)

smallmol_clr_paired.to_csv(
    OUT_DIR / "small_molecule_projected_cNMF_CLR_paired_replicate_differences.tsv",
    sep="\t",
    index=False,
)

display(smallmol_clr_effects.query("program in @KEY_PROGRAMS").head(30))

In [ ]:
pert = pd.read_excel(XLSX_PATH, sheet_name=PROGRAM_EFFECT_SHEET)

print(pert.columns.tolist())
display(pert.head())

required = ["target_name", "program_name", "Sample", PERTURBSEQ_EFFECT_COL]
missing = [c for c in required if c not in pert.columns]
if missing:
    raise ValueError(f"Missing columns from workbook sheet: {missing}")

pert = pert.copy()
pert["program"] = "P" + pert["program_name"].astype(int).astype(str)
pert = pert[pert["target_name"].isin(HIPPO_KD_TARGETS)].copy()
pert = pert[pert["Sample"].isin(["D2", "D3"])].copy()

hippo_ref = (
    pert
    .groupby(["Sample", "program"], observed=True)[PERTURBSEQ_EFFECT_COL]
    .mean()
    .reset_index()
    .rename(columns={
        "Sample": "perturbseq_reference_day",
        PERTURBSEQ_EFFECT_COL: "perturbseq_hippo_kd_effect",
    })
)

display(hippo_ref.head())

In [ ]:
concordance_records = []
merged_records = []

for contrast_name, sub_sm in smallmol_clr_effects.groupby("contrast", observed=True):
    for ref_day in ["D2", "D3"]:
        sub_ref = hippo_ref[hippo_ref["perturbseq_reference_day"] == ref_day]

        merged = sub_sm[["contrast", "treatment", "program", "mean_diff_clr"]].merge(
            sub_ref[["perturbseq_reference_day", "program", "perturbseq_hippo_kd_effect"]],
            on="program",
            how="inner",
        )

        if merged.shape[0] < 5:
            continue

        x = merged["perturbseq_hippo_kd_effect"].to_numpy(dtype=float)
        y = merged["mean_diff_clr"].to_numpy(dtype=float)

        pearson_r, pearson_p = pearsonr(x, y)
        spearman_r, spearman_p = spearmanr(x, y)

        concordance_records.append({
            "contrast": contrast_name,
            "treatment": sub_sm["treatment"].iloc[0],
            "perturbseq_reference_day": ref_day,
            "n_programs": merged.shape[0],
            "pearson_r": pearson_r,
            "pearson_pvalue": pearson_p,
            "spearman_r": spearman_r,
            "spearman_pvalue": spearman_p,
            "smallmol_effect_scale": f"CLR_logbase_{LOG_BASE}",
            "perturbseq_effect_col": PERTURBSEQ_EFFECT_COL,
        })

        merged_records.append(merged)

clr_concordance = pd.DataFrame(concordance_records)
clr_merged_values = pd.concat(merged_records, ignore_index=True)

clr_concordance.to_csv(
    OUT_DIR / "small_molecule_CLR_vs_perturbseq_HippoKD_D2_D3_concordance.tsv",
    sep="\t",
    index=False,
)

clr_merged_values.to_csv(
    OUT_DIR / "small_molecule_CLR_vs_perturbseq_HippoKD_D2_D3_program_values.tsv",
    sep="\t",
    index=False,
)

display(clr_concordance.sort_values("spearman_r", ascending=False))
display(clr_merged_values[clr_merged_values["program"].isin(KEY_PROGRAMS)].head(30))

In [ ]:
# %% [markdown]
# # Plot CLR-matched program concordance results
#
# Append these cells after the CLR concordance cells. They load the saved CLR result tables if the in-memory dataframes are not already present, then generate Illustrator-editable SVG/PDF/PNG plots.

# %%
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",  # editable text in Illustrator
})

BASE = Path("/scratch/users/edtng/20260325_Hippo_timecourse")
CLR_OUT_DIR = BASE / "perturbseq_cNMF_matched_DE" / "program_score_DE_clr"
CLR_FIG_DIR = CLR_OUT_DIR / "figures"
CLR_FIG_DIR.mkdir(parents=True, exist_ok=True)

KEY_PROGRAMS = ["P15", "P17", "P34"]
REFERENCE_DAY_ORDER = ["D2", "D3"]

CONTRAST_ORDER = [
    "D1_MPS+TDI",
    "D1_MPS+IAG",
    "D1.33_LM+TDI",
    "D1.33_LM+IAG",
    "D1.67_LM+TDI",
    "D1.67_LM+IAG",
    "D2_LM+TDI",
    "D2_LM+IAG",
    "D3_AE+TDI",
    "D3_AE+IAG",
    "D2_LM+TDI->D3_AE+TDI",
    "D2_LM+IAG->D3_AE+IAG",
]

# Load from disk if the analysis cells have already saved outputs.
def load_or_get(name, path):
    if name in globals():
        return globals()[name]
    if not Path(path).exists():
        raise FileNotFoundError(f"Could not find {path}; run the CLR concordance analysis cells first.")
    return pd.read_csv(path, sep="\t")

clr_concordance = load_or_get(
    "clr_concordance",
    CLR_OUT_DIR / "small_molecule_CLR_vs_perturbseq_HippoKD_D2_D3_concordance.tsv",
)

clr_merged_values = load_or_get(
    "clr_merged_values",
    CLR_OUT_DIR / "small_molecule_CLR_vs_perturbseq_HippoKD_D2_D3_program_values.tsv",
)

smallmol_clr_effects = load_or_get(
    "smallmol_clr_effects",
    CLR_OUT_DIR / "small_molecule_projected_cNMF_CLR_matched_effects.tsv",
)

smallmol_clr_paired = load_or_get(
    "smallmol_clr_paired",
    CLR_OUT_DIR / "small_molecule_projected_cNMF_CLR_paired_replicate_differences.tsv",
)

print("clr_concordance:", clr_concordance.shape)
print("clr_merged_values:", clr_merged_values.shape)
print("smallmol_clr_effects:", smallmol_clr_effects.shape)
print("smallmol_clr_paired:", smallmol_clr_paired.shape)

# %% [markdown]
# ## Plot 1: Spearman concordance heatmap against perturb-seq D2 and D3 Hippo KD

# %%
def _ordered_contrasts(df):
    observed = df["contrast"].drop_duplicates().tolist()
    ordered = [c for c in CONTRAST_ORDER if c in observed]
    ordered += [c for c in observed if c not in ordered]
    return ordered

contrast_order = _ordered_contrasts(clr_concordance)

heat = (
    clr_concordance
    .pivot_table(
        index="contrast",
        columns="perturbseq_reference_day",
        values="spearman_r",
        aggfunc="first",
    )
    .reindex(index=contrast_order, columns=REFERENCE_DAY_ORDER)
)

pval_heat = (
    clr_concordance
    .pivot_table(
        index="contrast",
        columns="perturbseq_reference_day",
        values="spearman_pvalue",
        aggfunc="first",
    )
    .reindex(index=contrast_order, columns=REFERENCE_DAY_ORDER)
)

mat = heat.to_numpy(dtype=float)
vmax = np.nanmax(np.abs(mat))

fig, ax = plt.subplots(figsize=(5.2, max(3.0, 0.34 * heat.shape[0] + 1)))
mesh = ax.pcolormesh(
    np.arange(heat.shape[1] + 1),
    np.arange(heat.shape[0] + 1),
    mat,
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
    shading="flat",
    linewidth=0.4,
    edgecolors="white",
)

ax.set_xlim(0, heat.shape[1])
ax.set_ylim(heat.shape[0], 0)
ax.set_xticks(np.arange(heat.shape[1]) + 0.5)
ax.set_xticklabels([f"Perturb-seq\nHippo KD {x}" for x in heat.columns])
ax.set_yticks(np.arange(heat.shape[0]) + 0.5)
ax.set_yticklabels(heat.index)
ax.set_title("CLR-matched concordance with perturb-seq Hippo KD")

for i, contrast in enumerate(heat.index):
    for j, day in enumerate(heat.columns):
        val = heat.loc[contrast, day]
        pval = pval_heat.loc[contrast, day]
        if pd.isna(val):
            continue
        star = ""
        if pd.notna(pval):
            if pval < 0.001:
                star = "***"
            elif pval < 0.01:
                star = "**"
            elif pval < 0.05:
                star = "*"
        ax.text(j + 0.5, i + 0.5, f"{val:.2f}{star}", ha="center", va="center", fontsize=7)

cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
cbar.set_label("Spearman rho across 50 programs")
fig.tight_layout()

for ext in ["svg", "pdf", "png"]:
    fig.savefig(CLR_FIG_DIR / f"CLR_concordance_heatmap_D2_D3_HippoKD.{ext}")
plt.show()

# %% [markdown]
# ## Plot 2: Spearman concordance bars

# %%
bar_df = clr_concordance.copy()
bar_df["contrast"] = pd.Categorical(bar_df["contrast"], categories=contrast_order, ordered=True)
bar_df["perturbseq_reference_day"] = pd.Categorical(
    bar_df["perturbseq_reference_day"],
    categories=REFERENCE_DAY_ORDER,
    ordered=True,
)
bar_df = bar_df.sort_values(["contrast", "perturbseq_reference_day"])

fig, ax = plt.subplots(figsize=(8.0, max(4.0, 0.36 * len(contrast_order))))
y_positions = []
y_labels = []
vals = []
colors = []

for i, contrast in enumerate(contrast_order):
    for offset, ref_day in zip([-0.16, 0.16], REFERENCE_DAY_ORDER):
        row = bar_df[(bar_df["contrast"] == contrast) & (bar_df["perturbseq_reference_day"] == ref_day)]
        if row.empty:
            continue
        y_positions.append(i + offset)
        y_labels.append(contrast)
        vals.append(float(row["spearman_r"].iloc[0]))
        colors.append("tab:orange" if ref_day == "D2" else "tab:blue")

ax.barh(y_positions, vals, height=0.28, color=colors)
ax.axvline(0, linewidth=0.8, color="black")
ax.set_yticks(range(len(contrast_order)))
ax.set_yticklabels(contrast_order)
ax.invert_yaxis()
ax.set_xlabel("Spearman rho across 50 programs")
ax.set_title("CLR-matched small molecule effect vs perturb-seq Hippo KD effect")

# manual legend
ax.scatter([], [], color="tab:orange", label="Perturb-seq D2 reference")
ax.scatter([], [], color="tab:blue", label="Perturb-seq D3 reference")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()

for ext in ["svg", "pdf", "png"]:
    fig.savefig(CLR_FIG_DIR / f"CLR_concordance_bars_D2_D3_HippoKD.{ext}")
plt.show()

# %% [markdown]
# ## Plot 3: Selected scatter panels with P15/P17/P34 highlighted

# %%
SELECTED_SCATTERS = [
    ("D2_LM+TDI", "D2"),
    ("D2_LM+IAG", "D2"),
    ("D3_AE+TDI", "D3"),
    ("D3_AE+IAG", "D3"),
    ("D2_LM+TDI->D3_AE+TDI", "D3"),
    ("D2_LM+IAG->D3_AE+IAG", "D3"),
]

n_panels = len(SELECTED_SCATTERS)
fig, axes = plt.subplots(2, 3, figsize=(11, 6.2), squeeze=False)
axes = axes.ravel()

for ax, (contrast, ref_day) in zip(axes, SELECTED_SCATTERS):
    sub = clr_merged_values[
        (clr_merged_values["contrast"] == contrast) &
        (clr_merged_values["perturbseq_reference_day"] == ref_day)
    ].copy()

    if sub.empty:
        ax.set_axis_off()
        ax.set_title(f"{contrast}\n{ref_day}: missing")
        continue

    x = sub["perturbseq_hippo_kd_effect"].to_numpy(dtype=float)
    y = sub["mean_diff_clr"].to_numpy(dtype=float)
    rho_row = clr_concordance[
        (clr_concordance["contrast"] == contrast) &
        (clr_concordance["perturbseq_reference_day"] == ref_day)
    ]
    rho = float(rho_row["spearman_r"].iloc[0]) if not rho_row.empty else np.nan

    ax.scatter(x, y, s=14, alpha=0.65, color="0.55")
    ax.axhline(0, linewidth=0.7, color="black")
    ax.axvline(0, linewidth=0.7, color="black")

    for program, color in zip(KEY_PROGRAMS, ["tab:red", "tab:green", "tab:orange"]):
        hit = sub[sub["program"] == program]
        if hit.empty:
            continue
        hx = float(hit["perturbseq_hippo_kd_effect"].iloc[0])
        hy = float(hit["mean_diff_clr"].iloc[0])
        ax.scatter([hx], [hy], s=45, color=color, edgecolor="black", linewidth=0.5, zorder=3)
        ax.text(hx, hy, f" {program}", va="center", fontsize=8)

    ax.set_title(f"{contrast}\nvs Hippo KD {ref_day}; rho={rho:.2f}")
    ax.set_xlabel("Perturb-seq Hippo KD program effect")
    ax.set_ylabel("Small molecule CLR program effect")

for ax in axes[n_panels:]:
    ax.set_axis_off()

fig.tight_layout()
for ext in ["svg", "pdf", "png"]:
    fig.savefig(CLR_FIG_DIR / f"CLR_concordance_scatter_selected_D2_D3_HippoKD.{ext}")
plt.show()

# %% [markdown]
# ## Plot 4: Key program CLR treatment effects

# %%
key_eff = smallmol_clr_effects[smallmol_clr_effects["program"].isin(KEY_PROGRAMS)].copy()
key_eff["contrast"] = pd.Categorical(key_eff["contrast"], categories=contrast_order, ordered=True)
key_eff["program"] = pd.Categorical(key_eff["program"], categories=KEY_PROGRAMS, ordered=True)
key_eff = key_eff.sort_values(["program", "contrast"])

heat_key = (
    key_eff
    .pivot_table(index="program", columns="contrast", values="mean_diff_clr", aggfunc="first")
    .reindex(index=KEY_PROGRAMS, columns=contrast_order)
)

mat = heat_key.to_numpy(dtype=float)
vmax = np.nanmax(np.abs(mat))

fig, ax = plt.subplots(figsize=(10.5, 2.2))
mesh = ax.pcolormesh(
    np.arange(heat_key.shape[1] + 1),
    np.arange(heat_key.shape[0] + 1),
    mat,
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
    shading="flat",
    linewidth=0.4,
    edgecolors="white",
)
ax.set_xlim(0, heat_key.shape[1])
ax.set_ylim(heat_key.shape[0], 0)
ax.set_xticks(np.arange(heat_key.shape[1]) + 0.5)
ax.set_xticklabels(heat_key.columns, rotation=45, ha="right")
ax.set_yticks(np.arange(heat_key.shape[0]) + 0.5)
ax.set_yticklabels(heat_key.index)
ax.set_title("Small molecule CLR effects on key projected cNMF programs")

for i, program in enumerate(heat_key.index):
    for j, contrast in enumerate(heat_key.columns):
        val = heat_key.loc[program, contrast]
        if pd.notna(val):
            ax.text(j + 0.5, i + 0.5, f"{val:.2f}", ha="center", va="center", fontsize=7)

cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
cbar.set_label("Matched mean CLR difference")
fig.tight_layout()

for ext in ["svg", "pdf", "png"]:
    fig.savefig(CLR_FIG_DIR / f"CLR_key_program_effect_heatmap.{ext}")
plt.show()

# %% [markdown]
# ## Plot 5: paired replicate CLR differences for P15/P17/P34

# %%
paired_key = smallmol_clr_paired[smallmol_clr_paired["program"].isin(KEY_PROGRAMS)].copy()
paired_key["contrast"] = pd.Categorical(paired_key["contrast"], categories=contrast_order, ordered=True)
paired_key["program"] = pd.Categorical(paired_key["program"], categories=KEY_PROGRAMS, ordered=True)
paired_key = paired_key.sort_values(["program", "contrast", "replicate"])

fig, axes = plt.subplots(1, len(KEY_PROGRAMS), figsize=(13, 3.4), sharey=True)
if len(KEY_PROGRAMS) == 1:
    axes = [axes]

for ax, program in zip(axes, KEY_PROGRAMS):
    sub = paired_key[paired_key["program"] == program]
    for j, contrast in enumerate(contrast_order):
        ss = sub[sub["contrast"] == contrast]
        if ss.empty:
            continue
        x = np.full(ss.shape[0], j, dtype=float)
        jitter = np.linspace(-0.08, 0.08, ss.shape[0]) if ss.shape[0] > 1 else np.array([0.0])
        vals = ss["paired_diff_clr"].to_numpy(dtype=float)
        ax.scatter(x + jitter, vals, s=22, color="black", alpha=0.8)
        ax.plot([j - 0.18, j + 0.18], [vals.mean(), vals.mean()], color="tab:red", linewidth=1.2)

    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_title(program)
    ax.set_xticks(np.arange(len(contrast_order)))
    ax.set_xticklabels(contrast_order, rotation=60, ha="right", fontsize=6)
    ax.set_ylabel("Treated - control CLR difference")

fig.suptitle("Paired replicate CLR differences for key projected cNMF programs", y=1.03)
fig.tight_layout()

for ext in ["svg", "pdf", "png"]:
    fig.savefig(CLR_FIG_DIR / f"CLR_key_program_paired_replicate_differences.{ext}")
plt.show()

# %%
print("Saved CLR concordance plots to:")
print(CLR_FIG_DIR)
for p in sorted(CLR_FIG_DIR.glob("*")):
    print(p.name)


#### Marker genes analysis

In [ ]:
# %% [markdown]
# # Hippo selected-program panels: D2/D3 programs, perturb-seq correlations, and marker genes
# 
# This notebook builds the additional Hippo validation panels discussed:
# 
# 1. **Selected-program effect heatmap** for D2 and D3 programs using matched small-molecule contrasts.
# 2. **Paired replicate plots** for selected day-relevant programs.
# 3. **Perturb-seq D2 × D3 program correlation matrix** using the `Perturbation Association 1` tab.
# 4. **Marker gene candidates** selected from top cNMF loading genes and concordant PyDESeq2 DE.
# 5. Optional **marker gene pseudobulk expression heatmap** if pseudobulk count files are available.
# 
# All SVG outputs keep text editable in Illustrator.

# %%
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr, fisher_exact
from statsmodels.stats.multitest import multipletests

try:
    from IPython.display import display
except Exception:
    def display(x):
        try:
            print(x.to_string())
        except Exception:
            print(x)

warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",  # Illustrator-editable text
})

# ---------- paths ----------
SEARCH_DIRS = [
    Path.cwd(),
    Path("/mnt/data"),
    Path("/scratch/users/edtng/20260325_Hippo_timecourse/perturbseq_cNMF_matched_DE/program_score_DE"),
    Path("/scratch/users/edtng/20260325_Hippo_timecourse/perturbseq_CNMf_matched_DE/program_score_DE"),
    Path("/scratch/users/edtng/20260325_Hippo_timecourse/perturbseq_cNMF_matched_DE/pydeseq2_gene_DE"),
]

def find_file(filename, extra_dirs=None, required=True):
    dirs = list(SEARCH_DIRS)
    if extra_dirs:
        dirs = list(extra_dirs) + dirs
    for d in dirs:
        p = d / filename
        if p.exists():
            return p
    if required:
        raise FileNotFoundError(f"Could not find {filename} in: {[str(d) for d in dirs]}")
    return None

# Uploaded/current files or Sherlock paths.
PROGRAM_EFFECTS_PATH = find_file("projected_cNMF_program_score_matched_contrasts.tsv")
PAIRED_DIFFS_PATH = find_file("projected_cNMF_program_score_paired_replicate_differences.tsv")
CNMF_XLSX_PATH = find_file("/scratch/users/edtng/20260325_Hippo_timecourse/cNMF_50_0_2_032526_6co.xlsx")

# Optional pseudobulk expression files for marker heatmap.
PSEUDOBULK_COUNTS_PATH = Path("/scratch/users/edtng/20260325_Hippo_timecourse/perturbseq_cNMF_matched_DE/pseudobulk_gene_counts.filtered.tsv.gz")
PSEUDOBULK_META_PATH = Path("/scratch/users/edtng/20260325_Hippo_timecourse/perturbseq_cNMF_matched_DE/pseudobulk_sample_metadata.filtered.tsv")

# Output folder. Use Sherlock folder when available, otherwise local folder.
DEFAULT_OUT = Path("/scratch/users/edtng/20260325_Hippo_timecourse/hippo_selected_program_panels")
OUT_DIR = DEFAULT_OUT if DEFAULT_OUT.parent.exists() else Path("hippo_selected_program_panels")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROGRAM_EFFECTS_PATH:", PROGRAM_EFFECTS_PATH)
print("PAIRED_DIFFS_PATH:", PAIRED_DIFFS_PATH)
print("CNMF_XLSX_PATH:", CNMF_XLSX_PATH)
print("OUT_DIR:", OUT_DIR.resolve())

# %%
# ---------- program sets ----------
# Anchor programs remain the central claim.
ANCHOR_PROGRAMS = ["P15", "P17", "P34"]

# Day-relevant supporting programs from peak-day/activity classification.
D2_PROGRAMS = ["P14", "P15", "P17", "P19", "P23"]
D3_EC_PROGRAMS = ["P2", "P3", "P8", "P11", "P42"]
D3_ALT_PROGRAMS = ["P16", "P24", "P27", "P33", "P34", "P35"]

SELECTED_PROGRAMS = D2_PROGRAMS + D3_EC_PROGRAMS + D3_ALT_PROGRAMS
SELECTED_PROGRAMS = list(dict.fromkeys(SELECTED_PROGRAMS))

PROGRAM_SET_MAP = {}
for p in D2_PROGRAMS: PROGRAM_SET_MAP[p] = "D2 active"
for p in D3_EC_PROGRAMS: PROGRAM_SET_MAP[p] = "D3 EC / vascular"
for p in D3_ALT_PROGRAMS: PROGRAM_SET_MAP[p] = "D3 alternative / mesenchymal"

# Fallback labels from your existing peak-day program map.
FALLBACK_PROGRAM_LABELS = {
    "P14": "lateral mesoderm",
    "P15": "endothelial / pre-EC",
    "P17": "epithelial-like",
    "P19": "posterior mesoderm",
    "P23": "EC induction",
    "P2": "EC migratory",
    "P3": "EC activated",
    "P8": "EC adhesion/Notch",
    "P11": "EC angiogenic",
    "P42": "vascular signaling",
    "P16": "cytoskeletal",
    "P24": "HOX patterning",
    "P27": "mesenchymal ECM",
    "P33": "mesenchymal-guidance",
    "P34": "contractile/YAP",
    "P35": "EMT/ECM",
}

# Contrasts for D2/D3-focused Hippo panel, using original names.
FOCAL_CONTRASTS = [
    "TDI_at_D2_LM",
    "IAG_at_D2_LM",
    "TDI_at_D3_AE",
    "IAG_at_D3_AE",
    "TDI_D2_to_D3_vs_D3_AE",
    "IAG_D2_to_D3_vs_D3_AE",
]

FOCAL_CONTRAST_LABELS = {
    "TDI_at_D2_LM": "D2_LM+TDI",
    "IAG_at_D2_LM": "D2_LM+IAG",
    "TDI_at_D3_AE": "D3_AE+TDI",
    "IAG_at_D3_AE": "D3_AE+IAG",
    "TDI_D2_to_D3_vs_D3_AE": "D2_LM+TDI→D3_AE+TDI",
    "IAG_D2_to_D3_vs_D3_AE": "D2_LM+IAG→D3_AE+IAG",
}

# For marker gene selection: expected directions for TDI claim.
# Direction refers to treated vs matched control.
MARKER_RULES = [
    {"program": "P15", "contrast_name": "TDI_at_D2_LM", "direction": "down", "label": "P15 down in D2 TDI"},
    {"program": "P17", "contrast_name": "TDI_at_D2_LM", "direction": "up", "label": "P17 up in D2 TDI"},
    {"program": "P34", "contrast_name": "TDI_at_D3_AE", "direction": "up", "label": "P34 up in D3 TDI"},
    {"program": "P34", "contrast_name": "TDI_D2_to_D3_vs_D3_AE", "direction": "up", "label": "P34 up in D2→D3 TDI"},
]

# Optional broader rules for D3 EC/alternative sets.
# Uncomment/add rules if you want marker genes for additional programs.
# for p in D3_EC_PROGRAMS:
#     MARKER_RULES.append({"program": p, "contrast_name": "TDI_at_D3_AE", "direction": "down", "label": f"{p} down in D3 TDI"})
# for p in D3_ALT_PROGRAMS:
#     MARKER_RULES.append({"program": p, "contrast_name": "TDI_at_D3_AE", "direction": "up", "label": f"{p} up in D3 TDI"})

print("Selected programs:", SELECTED_PROGRAMS)
print("Focal contrasts:", FOCAL_CONTRASTS)

# %%
# ---------- helpers ----------
def program_num(program):
    m = re.fullmatch(r"P(\d+)", str(program))
    if m is None:
        raise ValueError(f"Program must be P<number>, got {program!r}")
    return int(m.group(1))


def compact_label(program):
    label = FALLBACK_PROGRAM_LABELS.get(program, program)
    return f"{program} — {label}"


def save_figure(fig, out_prefix: Path):
    out_prefix.parent.mkdir(parents=True, exist_ok=True)
    for ext in ["svg", "pdf", "png"]:
        kwargs = {"bbox_inches": "tight"}
        if ext == "png":
            kwargs["dpi"] = 300
        fig.savefig(out_prefix.with_suffix(f".{ext}"), **kwargs)
    print("Saved:", out_prefix.with_suffix(".svg"))


def bh_adjust(pvals):
    pvals = pd.Series(pvals, dtype=float)
    ok = pvals.notna()
    out = pd.Series(np.nan, index=pvals.index, dtype=float)
    if ok.sum() > 0:
        out.loc[ok] = multipletests(pvals.loc[ok], method="fdr_bh")[1]
    return out


def zscore_rows(df):
    z = df.sub(df.mean(axis=1), axis=0)
    z = z.div(df.std(axis=1).replace(0, np.nan), axis=0)
    return z.fillna(0)


def plot_matrix_heatmap(df, title, out_prefix, cmap="RdBu_r", center_zero=True, annotate=False, vlim=None, x_rotation=45, y_fontsize=7):
    mat = df.to_numpy(dtype=float)
    n_rows, n_cols = mat.shape
    if vlim is not None:
        vmin, vmax = -vlim, vlim if center_zero else (np.nanmin(mat), vlim)
    elif center_zero:
        vmax = np.nanmax(np.abs(mat))
        vmin = -vmax
    else:
        vmin, vmax = np.nanmin(mat), np.nanmax(mat)
        if np.isclose(vmin, vmax):
            vmax = vmin + 1e-9

    fig_w = max(7, 0.75 * n_cols + 4)
    fig_h = max(5, 0.33 * n_rows + 2)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    mesh = ax.pcolormesh(
        np.arange(n_cols + 1), np.arange(n_rows + 1), mat,
        cmap=cmap, vmin=vmin, vmax=vmax, shading="flat", linewidth=0.3,
        edgecolors="white", antialiased=False,
    )
    ax.set_xlim(0, n_cols)
    ax.set_ylim(n_rows, 0)
    ax.set_xticks(np.arange(n_cols) + 0.5)
    ax.set_yticks(np.arange(n_rows) + 0.5)
    ax.set_xticklabels(df.columns.tolist(), rotation=x_rotation, ha="right")
    ax.set_yticklabels(df.index.tolist(), fontsize=y_fontsize)
    ax.set_title(title)
    ax.tick_params(length=0)
    for spine in ax.spines.values(): spine.set_visible(False)

    if annotate:
        for i in range(n_rows):
            for j in range(n_cols):
                val = mat[i, j]
                if np.isfinite(val):
                    ax.text(j + 0.5, i + 0.5, f"{val:.2f}", ha="center", va="center", fontsize=6)

    cbar = fig.colorbar(mesh, ax=ax, pad=0.02, fraction=0.035)
    fig.tight_layout()
    save_figure(fig, out_prefix)
    plt.show()
    plt.close(fig)

# %% [markdown]
# ## 1. Selected-program effect heatmap
# 
# This panel focuses on day-relevant programs rather than all 50. It uses the sample-level matched effect table, so values are treatment minus matched control after sample-level aggregation.

# %%
effects = pd.read_csv(PROGRAM_EFFECTS_PATH, sep="\t")
paired = pd.read_csv(PAIRED_DIFFS_PATH, sep="\t")

print("effects columns:", effects.columns.tolist())
print("paired columns:", paired.columns.tolist())
print(effects.head())

# Filter focal programs/contrasts.
sel = effects[
    effects["program"].isin(SELECTED_PROGRAMS) &
    effects["contrast_name"].isin(FOCAL_CONTRASTS)
].copy()

# Pivot to selected programs x focal contrasts.
effect_col = "diff_treated_minus_control"
effect_mat = sel.pivot_table(index="program", columns="contrast_name", values=effect_col, aggfunc="first")
effect_mat = effect_mat.reindex(index=SELECTED_PROGRAMS, columns=FOCAL_CONTRASTS)
effect_mat.columns = [FOCAL_CONTRAST_LABELS[c] for c in effect_mat.columns]
effect_mat.index = [compact_label(p) for p in effect_mat.index]

# Save source table.
effect_mat.to_csv(OUT_DIR / "selected_D2_D3_program_effect_matrix.tsv", sep="\t")

# Plot.
plot_matrix_heatmap(
    effect_mat,
    title="Selected D2/D3 projected cNMF program effects\nsmall molecule treated − matched control",
    out_prefix=OUT_DIR / "selected_D2_D3_program_effect_heatmap",
    cmap="RdBu_r",
    center_zero=True,
    annotate=True,
    x_rotation=45,
    y_fontsize=7,
)

display(effect_mat)

# %% [markdown]
# ## 2. Paired replicate plots for selected programs
# 
# These panels show biological-replicate pairing. They are useful for checking whether effects are consistent or driven by one replicate.

# %%
def plot_paired_lines_for_programs(programs, contrasts, out_prefix, title):
    n_rows = len(programs)
    n_cols = len(contrasts)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(max(10, 1.9*n_cols), max(5, 1.7*n_rows)), squeeze=False)

    for i, program in enumerate(programs):
        for j, contrast in enumerate(contrasts):
            ax = axes[i, j]
            sub = paired[(paired["program"] == program) & (paired["contrast_name"] == contrast)].copy()
            if sub.empty:
                ax.axis("off")
                continue

            for _, row in sub.iterrows():
                ax.plot([0, 1], [row["control_value"], row["treated_value"]], marker="o", linewidth=0.9, alpha=0.8)
                ax.text(1.03, row["treated_value"], str(row["replicate_de"]).replace("rep", "r"), fontsize=5, va="center")

            mean_control = sub["control_value"].mean()
            mean_treated = sub["treated_value"].mean()
            ax.plot([0, 1], [mean_control, mean_treated], marker="s", linewidth=2.0, color="black")
            ax.set_xlim(-0.25, 1.35)
            ax.set_xticks([0, 1])
            ax.set_xticklabels(["ctrl", "treated"], rotation=45, ha="right")
            if i == 0:
                ax.set_title(FOCAL_CONTRAST_LABELS.get(contrast, contrast), fontsize=7)
            if j == 0:
                ax.set_ylabel(compact_label(program), fontsize=7)
            ax.grid(axis="y", linewidth=0.3, alpha=0.3)
            for spine in ["top", "right"]: ax.spines[spine].set_visible(False)

    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    save_figure(fig, out_prefix)
    plt.show()
    plt.close(fig)

plot_paired_lines_for_programs(
    programs=SELECTED_PROGRAMS,
    contrasts=FOCAL_CONTRASTS,
    out_prefix=OUT_DIR / "selected_D2_D3_program_paired_replicate_lines",
    title="Selected D2/D3 projected cNMF program usage by matched replicate",
)

# %% [markdown]
# ## 3. Expanded perturb-seq D2 × D3 program correlation matrix
# 
# This tests the developmental relationship between D2-active programs and D3 endpoint programs using perturb-seq KD effects. Each correlation is computed across perturbation targets in the `Perturbation Association 1` tab.
# 
# Rows: D2 programs. Columns: D3 EC and D3 alternative programs.

# %%
PERT_SHEET = "Perturbation Association 1"
PERT_EFFECT_COL = "approx_log2FC"  # use beta_hat if you want natural-log beta scale
PERT_TARGET_COL = "target_name"

pert = pd.read_excel(CNMF_XLSX_PATH, sheet_name=PERT_SHEET)
required = [PERT_TARGET_COL, "program_name", "Sample", PERT_EFFECT_COL]
missing = [c for c in required if c not in pert.columns]
if missing:
    raise ValueError(f"Missing required columns in {PERT_SHEET}: {missing}")

pert = pert.copy()
pert["program"] = "P" + pert["program_name"].astype(int).astype(str)
pert[PERT_EFFECT_COL] = pd.to_numeric(pert[PERT_EFFECT_COL], errors="coerce")

D2_CORR_PROGRAMS = D2_PROGRAMS
D3_CORR_PROGRAMS = D3_EC_PROGRAMS + D3_ALT_PROGRAMS

corr_records = []
for p2 in D2_CORR_PROGRAMS:
    d2 = pert[(pert["Sample"].astype(str) == "D2") & (pert["program"] == p2)][[PERT_TARGET_COL, PERT_EFFECT_COL]].dropna()
    d2 = d2.rename(columns={PERT_EFFECT_COL: "D2_effect"}).drop_duplicates(PERT_TARGET_COL)
    for p3 in D3_CORR_PROGRAMS:
        d3 = pert[(pert["Sample"].astype(str) == "D3") & (pert["program"] == p3)][[PERT_TARGET_COL, PERT_EFFECT_COL]].dropna()
        d3 = d3.rename(columns={PERT_EFFECT_COL: "D3_effect"}).drop_duplicates(PERT_TARGET_COL)
        m = d2.merge(d3, on=PERT_TARGET_COL, how="inner")
        if m.shape[0] >= 5:
            rho, pval = spearmanr(m["D2_effect"], m["D3_effect"])
            pr, pp = pearsonr(m["D2_effect"], m["D3_effect"])
        else:
            rho, pval, pr, pp = np.nan, np.nan, np.nan, np.nan
        corr_records.append({
            "D2_program": p2,
            "D3_program": p3,
            "D3_program_class": PROGRAM_SET_MAP.get(p3, ""),
            "n_targets": m.shape[0],
            "spearman_r": rho,
            "spearman_pvalue": pval,
            "pearson_r": pr,
            "pearson_pvalue": pp,
        })

corr_df = pd.DataFrame(corr_records)
corr_df["spearman_padj_bh"] = bh_adjust(corr_df["spearman_pvalue"])
corr_df.to_csv(OUT_DIR / "perturbseq_D2_vs_D3_program_correlation_table.tsv", sep="\t", index=False)

corr_mat = corr_df.pivot(index="D2_program", columns="D3_program", values="spearman_r").reindex(index=D2_CORR_PROGRAMS, columns=D3_CORR_PROGRAMS)
corr_mat_labeled = corr_mat.copy()
corr_mat_labeled.index = [compact_label(p) for p in corr_mat_labeled.index]
corr_mat_labeled.columns = [compact_label(p) for p in corr_mat_labeled.columns]

plot_matrix_heatmap(
    corr_mat_labeled,
    title="Perturb-seq program effect correlations\nD2 active programs vs D3 endpoint programs across perturbations",
    out_prefix=OUT_DIR / "perturbseq_D2_vs_D3_program_correlation_heatmap",
    cmap="RdBu_r",
    center_zero=True,
    annotate=True,
    x_rotation=45,
    y_fontsize=7,
)

display(corr_df.sort_values("spearman_r", ascending=False).head(15))
display(corr_df.sort_values("spearman_r").head(15))

# %% [markdown]
# ## 4. Marker gene candidates from top cNMF loadings and concordant DE
# 
# This selects genes that are both top-loading in a program and DE in the expected direction in a small-molecule contrast. It creates a reproducible gene list for marker-expression panels.

# %%
LOADINGS_SHEET = "Program Loadings"
TOP_N_LOADING = 300
MAX_MARKERS_PER_RULE = 8
DE_PADJ_CUTOFF = 0.05
DE_ABS_LFC_MIN = 0.25

loadings = pd.read_excel(CNMF_XLSX_PATH, sheet_name=LOADINGS_SHEET)
required = ["Program", "Rank", "Gene"]
missing = [c for c in required if c not in loadings.columns]
if missing:
    raise ValueError(f"Missing columns in Program Loadings: {missing}")

loadings = loadings.copy()
loadings["program"] = "P" + loadings["Program"].astype(int).astype(str)
loadings["Rank"] = pd.to_numeric(loadings["Rank"], errors="coerce")
loadings["Gene"] = loadings["Gene"].astype(str)
loadings_top = loadings[loadings["Rank"] <= TOP_N_LOADING].copy()

# Locate PyDESeq2 result table for each contrast.
def find_deseq_file(contrast_name):
    filename = f"PyDESeq2_{contrast_name}.tsv"
    return find_file(filename, required=True)

candidate_records = []
for rule in MARKER_RULES:
    program = rule["program"]
    contrast_name = rule["contrast_name"]
    direction = rule["direction"]
    label = rule["label"]

    de_path = find_deseq_file(contrast_name)
    de = pd.read_csv(de_path, sep="\t")
    if "gene_symbol" not in de.columns or "log2FoldChange" not in de.columns or "padj" not in de.columns:
        raise ValueError(f"DE table lacks required columns: {de_path}")

    de = de.copy()
    de["gene_symbol"] = de["gene_symbol"].astype(str)
    de["log2FoldChange"] = pd.to_numeric(de["log2FoldChange"], errors="coerce")
    de["padj"] = pd.to_numeric(de["padj"], errors="coerce")

    top = loadings_top[loadings_top["program"] == program][["program", "Rank", "Gene"]].copy()
    merged = top.merge(de, left_on="Gene", right_on="gene_symbol", how="inner")

    if direction == "up":
        merged = merged[(merged["padj"] < DE_PADJ_CUTOFF) & (merged["log2FoldChange"] >= DE_ABS_LFC_MIN)]
        merged = merged.sort_values(["Rank", "padj", "log2FoldChange"], ascending=[True, True, False])
    elif direction == "down":
        merged = merged[(merged["padj"] < DE_PADJ_CUTOFF) & (merged["log2FoldChange"] <= -DE_ABS_LFC_MIN)]
        merged = merged.sort_values(["Rank", "padj", "log2FoldChange"], ascending=[True, True, True])
    else:
        raise ValueError("direction must be up or down")

    merged = merged.head(MAX_MARKERS_PER_RULE).copy()
    merged["marker_rule"] = label
    merged["program"] = program
    merged["contrast_name"] = contrast_name
    merged["expected_direction"] = direction
    candidate_records.append(merged)

marker_candidates = pd.concat(candidate_records, ignore_index=True) if candidate_records else pd.DataFrame()
keep_cols = [
    "marker_rule", "program", "Gene", "Rank", "contrast_name", "expected_direction",
    "gene_id", "gene_symbol", "log2FoldChange", "padj", "baseMean", "stat"
]
keep_cols = [c for c in keep_cols if c in marker_candidates.columns]
marker_candidates = marker_candidates[keep_cols]
marker_candidates.to_csv(OUT_DIR / "selected_program_marker_gene_candidates.tsv", sep="\t", index=False)

display(marker_candidates)
print("Saved:", OUT_DIR / "selected_program_marker_gene_candidates.tsv")

# %% [markdown]
# ## 5. Optional marker gene pseudobulk expression heatmap
# 
# This requires pseudobulk counts and sample metadata files. If those files are not present in the current environment, this cell will skip plotting after saving the marker candidate table.

# %%
if not PSEUDOBULK_COUNTS_PATH.exists() or not PSEUDOBULK_META_PATH.exists():
    print("Pseudobulk counts/meta not found. Skipping marker expression heatmap.")
    print("Expected counts:", PSEUDOBULK_COUNTS_PATH)
    print("Expected metadata:", PSEUDOBULK_META_PATH)
else:
    counts = pd.read_csv(PSEUDOBULK_COUNTS_PATH, sep="\t", index_col=0)
    sample_meta = pd.read_csv(PSEUDOBULK_META_PATH, sep="\t")
    if "sample_name" in sample_meta.columns:
        sample_col = "sample_name"
    elif "sample" in sample_meta.columns:
        sample_col = "sample"
    else:
        sample_col = sample_meta.columns[0]

    sample_meta = sample_meta.set_index(sample_col, drop=False)
    common_samples = [s for s in counts.columns if s in sample_meta.index]
    counts = counts[common_samples]
    sample_meta = sample_meta.loc[common_samples]

    marker_genes = marker_candidates["Gene"].drop_duplicates().tolist()
    marker_genes_present = [g for g in marker_genes if g in counts.index]
    print(f"Marker genes present in pseudobulk counts: {len(marker_genes_present)} / {len(marker_genes)}")

    lib_size = counts.sum(axis=0)
    cpm = counts.div(lib_size, axis=1) * 1e6
    log_cpm = np.log2(cpm + 1)

    expr = log_cpm.loc[marker_genes_present].copy()

    # Sample order: D2/D3 focused, using the sample names as they appear.
    preferred_conditions = [
        "D2_LM", "D2_LM+TDI", "D2_LM+IAG",
        "D3_AE", "D3_AE+TDI", "D3_AE+IAG",
        "D2_LM+TDI->D3_AE+TDI", "D2_LM+IAG->D3_AE+IAG",
    ]
    ordered_samples = []
    for cond in preferred_conditions:
        ordered_samples += [s for s in expr.columns if str(s).startswith(cond + "_") or str(s) == cond]
    ordered_samples += [s for s in expr.columns if s not in ordered_samples]
    expr = expr[ordered_samples]

    # Order genes by marker rule/program.
    gene_order = []
    gene_labels = []
    for _, row in marker_candidates.drop_duplicates("Gene").iterrows():
        gene = row["Gene"]
        if gene in expr.index:
            gene_order.append(gene)
            gene_labels.append(f"{row['program']} {gene}")
    expr = expr.loc[gene_order]
    expr.index = gene_labels

    expr_z = zscore_rows(expr)
    expr.to_csv(OUT_DIR / "selected_program_marker_gene_log2CPM_by_sample.tsv", sep="\t")
    expr_z.to_csv(OUT_DIR / "selected_program_marker_gene_log2CPM_zscore_by_gene.tsv", sep="\t")

    plot_matrix_heatmap(
        expr_z,
        title="Selected P15/P17/P34 marker genes\npseudobulk log2(CPM+1), z-scored by gene",
        out_prefix=OUT_DIR / "selected_program_marker_gene_expression_heatmap",
        cmap="RdBu_r",
        center_zero=True,
        annotate=False,
        x_rotation=60,
        y_fontsize=7,
    )

display(marker_candidates)

# %% [markdown]
# ## Interpretation guide
# 
# Use these outputs as follows:
# 
# - `selected_D2_D3_program_effect_heatmap.svg`: compact program effect panel for D2/D3 programs.
# - `selected_D2_D3_program_paired_replicate_lines.svg`: replicate-aware program score evidence.
# - `perturbseq_D2_vs_D3_program_correlation_heatmap.svg`: expanded D2-to-D3 program relationship map from perturb-seq effects.
# - `selected_program_marker_gene_candidates.tsv`: reproducible marker-gene set based on cNMF loadings and concordant DE.
# - `selected_program_marker_gene_expression_heatmap.svg`: marker expression heatmap if pseudobulk counts are available.
# 
# For the main figure, I would use the selected-program effect heatmap + a marker gene heatmap. For the supplement, use the paired replicate lines and full D2×D3 correlation matrix.